In [1]:
# Study v2 preflight: fetch and authenticate the prepared bundle; no extraction/training yet.
from pathlib import Path, PurePosixPath
import hashlib, subprocess, tarfile
ROOT=Path('/home/sagemaker-user/brain-mri-segmentation')
VERSIONDIR=ROOT/'versions'/'study_v2'
BUCKET='ajay-mlp1-lgg-044450716277-use1'
KEY='studio/study_v2/study_v2_bundle_20261108T1005Z/study_v2_bundle_20261108T1005Z.tar.gz'
EXPECTED_SIZE=30965
EXPECTED_SHA256='8e3577f43ebcd8d39b14912580b8806c4940c717e60b02328eca503f6e017f7d'
BUNDLE=Path('/tmp/study_v2_bundle_20261108T1005Z.tar.gz')
assert ROOT.resolve()==Path('/home/sagemaker-user/brain-mri-segmentation')
assert not VERSIONDIR.exists(),f'Refusing to overwrite version directory: {VERSIONDIR}'
if BUNDLE.exists():
    digest=hashlib.sha256(BUNDLE.read_bytes()).hexdigest()
    assert BUNDLE.stat().st_size==EXPECTED_SIZE and digest==EXPECTED_SHA256, 'Existing temp bundle differs; refusing overwrite'
    download_note='reused existing matching temp archive'
else:
    res=subprocess.run(['aws','s3','cp',f's3://{BUCKET}/{KEY}',str(BUNDLE)],check=True,capture_output=True,text=True)
    download_note=res.stdout.strip()
actual_size=BUNDLE.stat().st_size
actual_sha=hashlib.sha256(BUNDLE.read_bytes()).hexdigest()
assert actual_size==EXPECTED_SIZE,(actual_size,EXPECTED_SIZE)
assert actual_sha==EXPECTED_SHA256,(actual_sha,EXPECTED_SHA256)
with tarfile.open(BUNDLE,'r:gz') as tar:
    members=tar.getmembers()
    for m in members:
        name=m.name.replace('\\','/')
        path=PurePosixPath(name)
        assert not path.is_absolute() and '..' not in path.parts,f'unsafe archive path: {m.name}'
        assert '\\' not in m.name,f'backslash path rejected: {m.name}'
        assert m.isdir() or m.isfile(),f'non-regular archive member rejected: {m.name}'
    names=[m.name for m in members]
    top=sorted({PurePosixPath(n).parts[0] for n in names if PurePosixPath(n).parts})
    inv=[n for n in names if PurePosixPath(n).name.lower() in {'inventory.sha256','inventory.sha256.txt'}]
    print('DOWNLOAD',download_note)
    print('VERSIONDIR_ABSENT',not VERSIONDIR.exists())
    print('BUNDLE_PATH',BUNDLE,'SIZE_BYTES',actual_size,'SHA256',actual_sha)
    print('ARCHIVE_MEMBER_COUNT',len(members),'TOP_LEVEL',top)
    print('INVENTORY_CANDIDATES',inv)
    print('MEMBERS',[(m.name,'dir' if m.isdir() else 'file',m.size) for m in members])

DOWNLOAD Completed 30.2 KiB/30.2 KiB (300.8 KiB/s) with 1 file(s) remaining
download: s3://ajay-mlp1-lgg-044450716277-use1/studio/study_v2/study_v2_bundle_20261108T1005Z/study_v2_bundle_20261108T1005Z.tar.gz to ../../../../tmp/study_v2_bundle_20261108T1005Z.tar.gz
VERSIONDIR_ABSENT True
BUNDLE_PATH /tmp/study_v2_bundle_20261108T1005Z.tar.gz SIZE_BYTES 30965 SHA256 8e3577f43ebcd8d39b14912580b8806c4940c717e60b02328eca503f6e017f7d
ARCHIVE_MEMBER_COUNT 42 TOP_LEVEL ['study_v2_bundle_20261108T1005Z']
INVENTORY_CANDIDATES ['study_v2_bundle_20261108T1005Z/INVENTORY.sha256']
MEMBERS [('study_v2_bundle_20261108T1005Z', 'dir', 0), ('study_v2_bundle_20261108T1005Z/INVENTORY.sha256', 'file', 3074), ('study_v2_bundle_20261108T1005Z/README.md', 'file', 4915), ('study_v2_bundle_20261108T1005Z/configs', 'dir', 0), ('study_v2_bundle_20261108T1005Z/configs/base.yaml', 'file', 358), ('study_v2_bundle_20261108T1005Z/configs/bce_baseline.yaml', 'file', 387), ('study_v2_bundle_20261108T1005Z/configs/bce_dic

In [2]:
# Safe extraction and inventory SHA-256 verification; no dataset/holdout content is opened.
from pathlib import Path, PurePosixPath
import tarfile, hashlib, re, json
ROOT=Path('/home/sagemaker-user/brain-mri-segmentation')
VERSIONDIR=ROOT/'versions'/'study_v2'
EVIDENCE=ROOT/'evidence'/'study_v2'
BUNDLE=Path('/tmp/study_v2_bundle_20261108T1005Z.tar.gz')
PREFIX='study_v2_bundle_20261108T1005Z'
assert BUNDLE.is_file() and not VERSIONDIR.exists(), 'Missing bundle or version folder already exists; refusing overwrite'
with tarfile.open(BUNDLE,'r:gz') as tar:
    members=tar.getmembers()
    inv_members=[m for m in members if PurePosixPath(m.name).name=='INVENTORY.sha256']
    assert len(inv_members)==1, f'Expected one INVENTORY.sha256, found {len(inv_members)}'
    inventory_text=tar.extractfile(inv_members[0]).read().decode('utf-8')
    assert all(PurePosixPath(m.name).parts and PurePosixPath(m.name).parts[0]==PREFIX for m in members), 'Unexpected archive root layout'
    for m in members:
        p=PurePosixPath(m.name)
        assert not p.is_absolute() and '..' not in p.parts and '\\' not in m.name, f'Unsafe archive path: {m.name}'
        assert m.isdir() or m.isfile(), f'Non-regular member rejected: {m.name}'
    files=[m for m in members if m.isfile()]
    dataset_ext={'.tif','.tiff','.nii','.nii.gz'}
    dataset_members=[m.name for m in files if any(m.name.lower().endswith(s) for s in dataset_ext)]
    assert not dataset_members, f'Dataset-like files unexpectedly present: {dataset_members}'
    test_manifest_names=[m.name for m in files if 'test_manifest' in PurePosixPath(m.name).name.lower()]
    assert not test_manifest_names, f'Dataset test-manifest files unexpectedly present: {test_manifest_names}'
    VERSIONDIR.parent.mkdir(parents=True,exist_ok=True)
    VERSIONDIR.mkdir(parents=False,exist_ok=False)
    for m in members:
        p=PurePosixPath(m.name).relative_to(PREFIX)
        if not p.parts: continue
        dst=VERSIONDIR.joinpath(*p.parts)
        assert dst.resolve()==VERSIONDIR.resolve() or VERSIONDIR.resolve() in dst.resolve().parents, f'Extraction escaped target: {dst}'
        if m.isdir():
            dst.mkdir(parents=True,exist_ok=True)
        else:
            dst.parent.mkdir(parents=True,exist_ok=True)
            assert not dst.exists(), f'Refusing to overwrite extracted file: {dst}'
            src=tar.extractfile(m)
            assert src is not None
            with dst.open('xb') as f: f.write(src.read())
inv_path=VERSIONDIR/'INVENTORY.sha256'
assert inv_path.is_file()
verified=[]; listed=set(); self_entries=[]
for line in inventory_text.splitlines():
    line=line.strip()
    if not line or line.startswith('#'): continue
    parts=line.split(None,1)
    assert len(parts)==2, f'Unrecognized inventory line: {line!r}'
    expected,name=parts[0].lower(),parts[1].lstrip('*').strip()
    assert re.fullmatch(r'[0-9a-f]{64}',expected), f'Invalid SHA-256 entry: {line!r}'
    rel=PurePosixPath(name)
    if rel.parts and rel.parts[0]==PREFIX: rel=rel.relative_to(PREFIX)
    assert not rel.is_absolute() and '..' not in rel.parts, f'Unsafe inventory path: {name}'
    if rel.as_posix()=='INVENTORY.sha256':
        self_entries.append(name); continue
    target=VERSIONDIR.joinpath(*rel.parts)
    assert target.is_file(), f'Inventory file missing after extraction: {name}'
    got=hashlib.sha256(target.read_bytes()).hexdigest()
    assert got==expected, f'Inventory mismatch for {name}: expected {expected}, got {got}'
    listed.add(rel.as_posix()); verified.append((rel.as_posix(),got))
actual={p.relative_to(VERSIONDIR).as_posix() for p in VERSIONDIR.rglob('*') if p.is_file() and p!=inv_path}
unlisted=sorted(actual-listed); missing=sorted(listed-actual)
assert not unlisted and not missing, {'unlisted':unlisted,'missing':missing}
roots=sorted(p.name for p in VERSIONDIR.iterdir())
report={'bundle_path':str(BUNDLE),'bundle_size_bytes':BUNDLE.stat().st_size,'bundle_sha256':hashlib.sha256(BUNDLE.read_bytes()).hexdigest(),'version_dir':str(VERSIONDIR),'inventory_path':str(inv_path),'verified_inventory_file_count':len(verified),'inventory_self_entry_skipped':bool(self_entries),'root_entries':roots,'dataset_files_in_bundle':dataset_members,'test_manifest_files_in_bundle':test_manifest_names}
EVIDENCE.mkdir(parents=True,exist_ok=True)
(EVIDENCE/'bundle_verification.json').write_text(json.dumps(report,indent=2,sort_keys=True)+'\n')
print('BUNDLE_DEPLOYMENT_VERIFIED')
print(json.dumps(report,indent=2,sort_keys=True))

BUNDLE_DEPLOYMENT_VERIFIED
{
  "bundle_path": "/tmp/study_v2_bundle_20261108T1005Z.tar.gz",
  "bundle_sha256": "8e3577f43ebcd8d39b14912580b8806c4940c717e60b02328eca503f6e017f7d",
  "bundle_size_bytes": 30965,
  "dataset_files_in_bundle": [],
  "inventory_path": "/home/sagemaker-user/brain-mri-segmentation/versions/study_v2/INVENTORY.sha256",
  "inventory_self_entry_skipped": false,
  "root_entries": [
    "INVENTORY.sha256",
    "README.md",
    "configs",
    "docs",
    "notebooks",
    "requirements-gpu.txt",
    "requirements.txt",
    "scripts",
    "src",
    "tests"
  ],
  "test_manifest_files_in_bundle": [],
  "verified_inventory_file_count": 33,
  "version_dir": "/home/sagemaker-user/brain-mri-segmentation/versions/study_v2"
}


In [3]:
# Read-only audit of the staged training-plan notebook/config paths; no plan cell is executed.
from pathlib import Path
import json,re,sys,os
import torch
ROOT=Path('/home/sagemaker-user/brain-mri-segmentation')
VERSIONDIR=ROOT/'versions'/'study_v2'
PLAN=VERSIONDIR/'notebooks'/'03_planned_study_v2_BATCH_DO_NOT_AUTORUN.ipynb'
assert PLAN.is_file()
nb=json.loads(PLAN.read_text())
print('PLAN_NOTEBOOK',PLAN)
print('PLAN_CELL_COUNT',len(nb.get('cells',[])))
path_pattern=re.compile(r'ROOT|VERSION|DATA|RUNS|OUT|cwd|PYTHONPATH|seed|config|epochs|run_count|run_name|output',re.I)
for idx,cell in enumerate(nb.get('cells',[])):
    if cell.get('cell_type')!='code': continue
    src=''.join(cell.get('source',[]))
    selected=[line.rstrip()[:260] for line in src.splitlines() if path_pattern.search(line)]
    if selected:
        print('PLAN_CODE_CELL',idx,'EXECUTION_COUNT',cell.get('execution_count'))
        for line in selected: print('  '+line)
print('CURRENT_NOTEBOOK_KERNEL_PYTHON',sys.executable)
print('CURRENT_TORCH',torch.__version__,'CUDA_BUILD',torch.version.cuda,'CUDA_AVAILABLE',torch.cuda.is_available())
print('EVIDENCE_FILES_EXISTING',[p.name for p in sorted((ROOT/'evidence'/'study_v2').iterdir())])

PLAN_NOTEBOOK /home/sagemaker-user/brain-mri-segmentation/versions/study_v2/notebooks/03_planned_study_v2_BATCH_DO_NOT_AUTORUN.ipynb
PLAN_CELL_COUNT 6
PLAN_CODE_CELL 1 EXECUTION_COUNT None
  ROOT=os.path.expanduser('~/brain-mri-segmentation'); DATA=f'{ROOT}/data'; OUT=f'{ROOT}/runs/study_v2'
  M=json.load(open(f'{ROOT}/docs/study_v2_execution_manifest.json'))
  assert not os.path.exists(f'{DATA}/test') and not os.path.exists(f'{DATA}/test_manifest.csv'), 'holdout must not be staged before freeze'
  subprocess.run([sys.executable,'-m','pytest','-q','tests'],cwd=ROOT,check=True)
      rd=f'{OUT}/{c}_s{s}'
      subprocess.run([sys.executable,'scripts/train.py','--config',f'configs/{c}.yaml','--data-root',DATA,'--out-root',OUT,'--seed',str(s)],cwd=ROOT,check=True)
  PLAN=[(r['config'].split('/')[-1][:-5],r['seed']) for r in M['runs']]; print(len(PLAN),PLAN)
PLAN_CODE_CELL 2 EXECUTION_COUNT None
  # MAIN: bce_baseline vs bce_dice, seeds 0,1,2 (6 runs)
PLAN_CODE_CELL 3 EXECUTION_COUNT None


In [4]:
# Create a copy of the bundled plan with explicit versioned paths; do not execute any plan cell.
from pathlib import Path
import json,re,os
ROOT=Path('/home/sagemaker-user/brain-mri-segmentation')
VERSIONDIR=ROOT/'versions'/'study_v2'
SRC=VERSIONDIR/'notebooks'/'03_planned_study_v2_BATCH_DO_NOT_AUTORUN.ipynb'
DEST=ROOT/'notebooks'/'03_study_v2_BATCH_DO_NOT_AUTORUN.ipynb'
DATA=ROOT/'data'; OUT=ROOT/'runs'/'study_v2'
assert SRC.is_file() and not DEST.exists(), f'Missing staged plan or refusing overwrite: {DEST}'
nb=json.loads(SRC.read_text())
old="ROOT=os.path.expanduser('~/brain-mri-segmentation'); DATA=f'{ROOT}/data'; OUT=f'{ROOT}/runs/study_v2'"
new="ROOT=os.path.abspath('/home/sagemaker-user/brain-mri-segmentation'); VERSIONDIR=os.path.join(ROOT,'versions','study_v2'); DATA=os.path.join(ROOT,'data'); OUT=os.path.join(ROOT,'runs','study_v2'); ENV=os.environ.copy(); ENV['PYTHONPATH']=os.path.join(VERSIONDIR,'src')"
patched_root=0; patched_manifest=0; patched_cwd=0
for cell in nb.get('cells',[]):
    src=''.join(cell.get('source',[])) if isinstance(cell.get('source'),list) else cell.get('source','')
    if old in src:
        src=src.replace(old,new); patched_root+=1
    src2=src.replace("f'{ROOT}/docs/study_v2_execution_manifest.json'","f'{VERSIONDIR}/docs/study_v2_execution_manifest.json'")
    if src2!=src: patched_manifest+=1; src=src2
    src2=src.replace('cwd=ROOT,check=True','cwd=VERSIONDIR,env=ENV,check=True')
    if src2!=src: patched_cwd+=src.count('cwd=ROOT,check=True'); src=src2
    if isinstance(cell.get('source'),list): cell['source']=src.splitlines(keepends=True)
    else: cell['source']=src
    if cell.get('cell_type')=='code':
        cell['execution_count']=None; cell['outputs']=[]
assert patched_root==1,(patched_root,'expected exactly one root setup path')
assert patched_manifest==1,(patched_manifest,'expected one execution-manifest path')
assert patched_cwd>=2,(patched_cwd,'expected pytest and training subprocess cwd updates')
comments=[]
for cell in nb.get('cells',[]):
    src=''.join(cell.get('source',[])) if isinstance(cell.get('source'),list) else cell.get('source','')
    comments += [ln.strip() for ln in src.splitlines() if re.search(r'#\s*(MAIN|DIAGNOSTIC|SENSITIVITY):',ln)]
counts=[int(m.group(1)) for s in comments if (m:=re.search(r'\((\d+)\s+runs?\)',s))]
assert sum(counts)==14,(comments,counts)
assert all('seeds 0,1,2' in s for s in comments[:2]),comments
configs=['bce_baseline.yaml','bce_dice.yaml','weighted_bce.yaml','bce_dice_posonly.yaml','bce_dice_w0.5.yaml','bce_dice_w2.yaml']
missing=[c for c in configs if not (VERSIONDIR/'configs'/c).is_file()]
assert not missing,missing
note={'cell_type':'markdown','metadata':{},'id':'study-v2-path-note','source':['# Study v2 training plan — path-corrected copy (DO NOT RUN)\n','\n','This copy is staged for review only. The training batch loops were not executed.\n','\n','- ROOT: /home/sagemaker-user/brain-mri-segmentation\n','- VERSIONDIR and subprocess cwd: ROOT/versions/study_v2\n','- PYTHONPATH: VERSIONDIR/src\n','- train/val data root: ROOT/data (existing train/val only)\n','- output root: ROOT/runs/study_v2\n','- planned grid: 6 main + 6 diagnostic + 2 sensitivity = 14 runs; seeds 0, 1, 2.\n','']}
nb['cells'].insert(0,note)
DEST.write_text(json.dumps(nb,indent=1,ensure_ascii=False)+'\n')
assert DEST.is_file()
report={'source_notebook':str(SRC),'new_notebook':str(DEST),'patched_root_setup_cells':patched_root,'patched_execution_manifest_paths':patched_manifest,'patched_subprocess_cwds':patched_cwd,'plan_group_comments':comments,'plan_run_counts':counts,'total_planned_runs':sum(counts),'expected_configs':configs,'missing_configs':missing,'train_manifest_exists':(DATA/'train_manifest.csv').is_file(),'val_manifest_exists':(DATA/'val_manifest.csv').is_file(),'output_root':str(OUT),'version_dir':str(VERSIONDIR),'plan_cells_executed':False}
print('PLAN_COPY_CREATED',json.dumps(report,indent=2))

PLAN_COPY_CREATED {
  "source_notebook": "/home/sagemaker-user/brain-mri-segmentation/versions/study_v2/notebooks/03_planned_study_v2_BATCH_DO_NOT_AUTORUN.ipynb",
  "new_notebook": "/home/sagemaker-user/brain-mri-segmentation/notebooks/03_study_v2_BATCH_DO_NOT_AUTORUN.ipynb",
  "patched_root_setup_cells": 1,
  "patched_execution_manifest_paths": 1,
  "patched_subprocess_cwds": 2,
  "plan_group_comments": [
    "# MAIN: bce_baseline vs bce_dice, seeds 0,1,2 (6 runs)",
    "# DIAGNOSTIC: weighted_bce, bce_dice_posonly, seeds 0,1,2 (6 runs)",
    "# SENSITIVITY: dice weight 0.5 and 2, seed 0 (2 runs)"
  ],
  "plan_run_counts": [
    6,
    6,
    2
  ],
  "total_planned_runs": 14,
  "expected_configs": [
    "bce_baseline.yaml",
    "bce_dice.yaml",
    "weighted_bce.yaml",
    "bce_dice_posonly.yaml",
    "bce_dice_w0.5.yaml",
    "bce_dice_w2.yaml"
  ],
  "missing_configs": [],
  "train_manifest_exists": true,
  "val_manifest_exists": true,
  "output_root": "/home/sagemaker-user/brain-m

In [5]:
# Run version-isolated preflight tests in a subprocess; no training or holdout access.
from pathlib import Path
import os,sys,json,subprocess,datetime,hashlib,re
import torch
ROOT=Path('/home/sagemaker-user/brain-mri-segmentation')
VERSIONDIR=ROOT/'versions'/'study_v2'
DATA=ROOT/'data'
OUT=ROOT/'runs'/'study_v2'
EVIDENCE=ROOT/'evidence'/'study_v2'
PLAN_COPY=ROOT/'notebooks'/'03_study_v2_BATCH_DO_NOT_AUTORUN.ipynb'
assert VERSIONDIR.is_dir() and (VERSIONDIR/'src').is_dir() and (VERSIONDIR/'tests').is_dir()
assert PLAN_COPY.is_file()
trainval={'train_manifest_exists':(DATA/'train_manifest.csv').is_file(),'val_manifest_exists':(DATA/'val_manifest.csv').is_file()}
env=os.environ.copy(); env['PYTHONPATH']=str(VERSIONDIR/'src'); env['PYTHONDONTWRITEBYTECODE']='1'
probe_code="import json,os,sys,torch,lggseg; print(json.dumps({'python':sys.executable,'cwd':os.getcwd(),'pythonpath':os.environ.get('PYTHONPATH'),'torch':torch.__version__,'cuda_build':torch.version.cuda,'cuda_available':torch.cuda.is_available(),'gpu':torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,'lggseg_file':lggseg.__file__}))"
probe_cmd=[sys.executable,'-c',probe_code]
probe=subprocess.run(probe_cmd,cwd=str(VERSIONDIR),env=env,capture_output=True,text=True)
probe_obj=None
if probe.returncode==0 and probe.stdout.strip():
    try: probe_obj=json.loads(probe.stdout.strip().splitlines()[-1])
    except Exception: probe_obj=None
version_src=str((VERSIONDIR/'src').resolve())
import_path_ok=bool(probe_obj and str(Path(probe_obj['lggseg_file']).resolve()).startswith(version_src+os.sep))
pytest_cmd=[sys.executable,'-m','pytest','-q','-p','no:cacheprovider','tests']
started=datetime.datetime.now(datetime.timezone.utc)
pytest_res=None
if probe.returncode==0 and import_path_ok:
    pytest_res=subprocess.run(pytest_cmd,cwd=str(VERSIONDIR),env=env,capture_output=True,text=True)
finished=datetime.datetime.now(datetime.timezone.utc)
pytest_stdout=pytest_res.stdout if pytest_res else ''
pytest_stderr=pytest_res.stderr if pytest_res else ''
pytest_rc=pytest_res.returncode if pytest_res else None
match=re.search(r'(?m)(\d+)\s+passed',pytest_stdout)
test_count=int(match.group(1)) if match else None
stamp=finished.strftime('%Y%m%dT%H%M%SZ')
plan=json.loads(PLAN_COPY.read_text())
plan_sources=[''.join(c.get('source',[])) if isinstance(c.get('source'),list) else c.get('source','') for c in plan.get('cells',[])]
plan_code='\n'.join(s for c,s in zip(plan.get('cells',[]),plan_sources) if c.get('cell_type')=='code')
comments=[ln.strip() for s in plan_sources for ln in s.splitlines() if re.search(r'#\s*(MAIN|DIAGNOSTIC|SENSITIVITY):',ln)]
run_counts=[int(m.group(1)) for s in comments if (m:=re.search(r'\((\d+)\s+runs?\)',s))]
assert sum(run_counts)==14 and 'cwd=ROOT' not in plan_code and "f'{ROOT}/docs/study_v2_execution_manifest.json'" not in plan_code
assert plan_code.count('cwd=VERSIONDIR,env=ENV,check=True')>=2
configs=['bce_baseline.yaml','bce_dice.yaml','weighted_bce.yaml','bce_dice_posonly.yaml','bce_dice_w0.5.yaml','bce_dice_w2.yaml']
assert all((VERSIONDIR/'configs'/c).is_file() for c in configs)
bundle_report=json.loads((EVIDENCE/'bundle_verification.json').read_text())
plan_audit={'plan_copy':str(PLAN_COPY),'source_plan':str(VERSIONDIR/'notebooks'/'03_planned_study_v2_BATCH_DO_NOT_AUTORUN.ipynb'),'path_setup_verified':True,'root':str(ROOT),'version_dir_and_subprocess_cwd':str(VERSIONDIR),'pythonpath':str(VERSIONDIR/'src'),'trainval_data_root':str(DATA),'trainval_manifest_files_exist':trainval,'output_root':str(OUT),'config_files':configs,'plan_groups':comments,'planned_run_counts':run_counts,'total_planned_runs':sum(run_counts),'plan_cells_executed':False}
env_report={'snapshot_utc':finished.isoformat(),'studio_kernel_python':sys.executable,'studio_kernel_torch':torch.__version__,'studio_kernel_cuda_build':torch.version.cuda,'studio_kernel_cuda_available':torch.cuda.is_available(),'studio_kernel_gpu':torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,'version_import_probe_command':probe_cmd,'version_import_probe_cwd':str(VERSIONDIR),'version_import_probe_pythonpath':env['PYTHONPATH'],'version_import_probe_returncode':probe.returncode,'version_import_probe_json':probe_obj,'version_import_path_ok':import_path_ok,'pytest_command':pytest_cmd,'pytest_cwd':str(VERSIONDIR),'pytest_pythonpath':env['PYTHONPATH'],'pytest_started_utc':started.isoformat(),'pytest_finished_utc':finished.isoformat(),'pytest_duration_s':(finished-started).total_seconds(),'pytest_returncode':pytest_rc,'pytest_passed_count':test_count,'pytest_stdout':pytest_stdout,'pytest_stderr':pytest_stderr,'bundle_sha256':bundle_report['bundle_sha256'],'bundle_size_bytes':bundle_report['bundle_size_bytes'],'inventory_verified_files':bundle_report['verified_inventory_file_count']}
report={'timestamp_utc':finished.isoformat(),'bundle':bundle_report,'environment':{k:v for k,v in env_report.items() if k not in {'pytest_stdout','pytest_stderr'}},'plan_audit':plan_audit,'pytest_success':bool(pytest_res and pytest_rc==0 and test_count==28),'test_count_expected':28,'no_training_launched':True,'no_test_or_holdout_metadata_images_or_masks_read':True,'training_output_root_not_created':not OUT.exists()}
EVIDENCE.mkdir(parents=True,exist_ok=True)
paths={'environment':EVIDENCE/f'preflight_environment_{stamp}.json','pytest_log':EVIDENCE/f'pytest_{stamp}.txt','plan_audit':EVIDENCE/f'plan_path_audit_{stamp}.json','report':EVIDENCE/f'preflight_report_{stamp}.json'}
for p in paths.values(): assert not p.exists(),f'Refusing to overwrite evidence: {p}'
for name,obj in [('environment',env_report),('plan_audit',plan_audit),('report',report)]:
    with paths[name].open('x') as f: json.dump(obj,f,indent=2,sort_keys=True); f.write('\n')
log='COMMAND: '+' '.join(pytest_cmd)+'\nCWD: '+str(VERSIONDIR)+'\nPYTHONPATH: '+env['PYTHONPATH']+'\nRETURN_CODE: '+str(pytest_rc)+'\nSTARTED_UTC: '+started.isoformat()+'\nFINISHED_UTC: '+finished.isoformat()+'\nSTDOUT:\n'+pytest_stdout+'\nSTDERR:\n'+pytest_stderr
with paths['pytest_log'].open('x') as f: f.write(log)
print('PREFLIGHT_DONE',json.dumps({'report':report,'evidence_paths':{k:str(v) for k,v in paths.items()}},indent=2))
print('PYTEST_STDOUT_BEGIN\n'+pytest_stdout+'\nPYTEST_STDERR_BEGIN\n'+pytest_stderr)

PREFLIGHT_DONE {
  "report": {
    "timestamp_utc": "2026-11-08T10:31:20.766514+00:00",
    "bundle": {
      "bundle_path": "/tmp/study_v2_bundle_20261108T1005Z.tar.gz",
      "bundle_sha256": "8e3577f43ebcd8d39b14912580b8806c4940c717e60b02328eca503f6e017f7d",
      "bundle_size_bytes": 30965,
      "dataset_files_in_bundle": [],
      "inventory_path": "/home/sagemaker-user/brain-mri-segmentation/versions/study_v2/INVENTORY.sha256",
      "inventory_self_entry_skipped": false,
      "root_entries": [
        "INVENTORY.sha256",
        "README.md",
        "configs",
        "docs",
        "notebooks",
        "requirements-gpu.txt",
        "requirements.txt",
        "scripts",
        "src",
        "tests"
      ],
      "test_manifest_files_in_bundle": [],
      "verified_inventory_file_count": 33,
      "version_dir": "/home/sagemaker-user/brain-mri-segmentation/versions/study_v2"
    },
    "environment": {
      "snapshot_utc": "2026-11-08T10:31:20.766514+00:00",
      "stud

In [6]:
# Preserve a timestamped evidence copy of the corrected plan and executed preflight notebook.
from pathlib import Path
import json,hashlib
ROOT=Path('/home/sagemaker-user/brain-mri-segmentation')
EVIDENCE=ROOT/'evidence'/'study_v2'
STAMP='20261108T103120Z'
PLAN=ROOT/'notebooks'/'03_study_v2_BATCH_DO_NOT_AUTORUN.ipynb'
PREFLIGHT=ROOT/'notebooks'/'study_v2_preflight.ipynb'
assert PLAN.is_file() and PREFLIGHT.is_file() and EVIDENCE.is_dir()
report=json.loads((EVIDENCE/f'preflight_report_{STAMP}.json').read_text())
copy_plan=EVIDENCE/f'03_study_v2_BATCH_DO_NOT_AUTORUN_{STAMP}.ipynb'
copy_preflight=EVIDENCE/f'study_v2_preflight_{STAMP}.ipynb'
readme=EVIDENCE/'README.md'
for p in (copy_plan,copy_preflight,readme): assert not p.exists(),f'Refusing to overwrite {p}'
for src,dst in ((PLAN,copy_plan),(PREFLIGHT,copy_preflight)):
    with dst.open('xb') as f: f.write(src.read_bytes())
b=report['bundle']; e=report['environment']; pa=report['plan_audit']
text=(
'# Study v2 deployment and preflight evidence\n\n'
+'Snapshot UTC: '+report['timestamp_utc']+'\n\n'
+'- Bundle S3 object: s3://ajay-mlp1-lgg-044450716277-use1/studio/study_v2/study_v2_bundle_20261108T1005Z/study_v2_bundle_20261108T1005Z.tar.gz\n'
+'- Bundle size/SHA-256: '+str(b['bundle_size_bytes'])+' bytes / '+b['bundle_sha256']+'\n'
+'- Safe extraction: '+b['version_dir']+'; verified inventory files: '+str(b['verified_inventory_file_count'])+'\n'
+'- Bundle dataset TIFFs/test manifests found: '+str(len(b['dataset_files_in_bundle']))+'/'+str(len(b['test_manifest_files_in_bundle']))+'\n'
+'- Preflight: '+str(e['pytest_passed_count'])+' tests passed, return code '+str(e['pytest_returncode'])+'; runtime '+str(e['pytest_duration_s'])+' s.\n'
+'- Environment: torch '+e['studio_kernel_torch']+', CUDA build '+e['studio_kernel_cuda_build']+', GPU '+str(e['studio_kernel_gpu'])+'.\n'
+'- Test subprocess cwd: '+e['pytest_cwd']+'\n'
+'- PYTHONPATH: '+e['pytest_pythonpath']+' (import resolved to versioned source).\n'
+'- Train/val data root: '+pa['trainval_data_root']+'; train_manifest.csv and val_manifest.csv exist (content not opened here).\n'
+'- Output root: '+pa['output_root']+'; 14-run grid is documented, not launched.\n'
+'- No test/holdout manifest content, images, or masks were read; no training loop was executed.\n\n'
+'Files: bundle_verification.json; preflight_environment_'+STAMP+'.json; pytest_'+STAMP+'.txt; plan_path_audit_'+STAMP+'.json; preflight_report_'+STAMP+'.json; '+copy_plan.name+'; '+copy_preflight.name+'; this README.md.\n'
)
with readme.open('x') as f: f.write(text)
items=[{'path':str(p),'size_bytes':p.stat().st_size,'sha256':hashlib.sha256(p.read_bytes()).hexdigest()} for p in (copy_plan,copy_preflight,readme)]
print('EVIDENCE_INDEX_SAVED',json.dumps({'evidence_dir':str(EVIDENCE),'copies':items,'readme':str(readme)},indent=2))

EVIDENCE_INDEX_SAVED {
  "evidence_dir": "/home/sagemaker-user/brain-mri-segmentation/evidence/study_v2",
  "copies": [
    {
      "path": "/home/sagemaker-user/brain-mri-segmentation/evidence/study_v2/03_study_v2_BATCH_DO_NOT_AUTORUN_20261108T103120Z.ipynb",
      "size_bytes": 3976,
      "sha256": "bb922f45e03454d1dd526f3794c624650e89acde418073f940986dbab786c98f"
    },
    {
      "path": "/home/sagemaker-user/brain-mri-segmentation/evidence/study_v2/study_v2_preflight_20261108T103120Z.ipynb",
      "size_bytes": 38361,
      "sha256": "c9f4af17ef2baad570d86ed0ccd84238d16827428c988d0d87edf075695a480a"
    },
    {
      "path": "/home/sagemaker-user/brain-mri-segmentation/evidence/study_v2/README.md",
      "size_bytes": 1488,
      "sha256": "7e23bcbae1999d5236bdfb93207871d3a1b70ec8d7c836f1c6dd6aef36757890"
    }
  ],
  "readme": "/home/sagemaker-user/brain-mri-segmentation/evidence/study_v2/README.md"
}


In [7]:
# Final compact receipt and inventory-manifest digest; all reads are of bundle/evidence files only.
from pathlib import Path
import json,hashlib,re
ROOT=Path('/home/sagemaker-user/brain-mri-segmentation')
VERSIONDIR=ROOT/'versions'/'study_v2'
EVIDENCE=ROOT/'evidence'/'study_v2'
STAMP='20261108T103120Z'
report=json.loads((EVIDENCE/f'preflight_report_{STAMP}.json').read_text())
inv=VERSIONDIR/'INVENTORY.sha256'
inv_bytes=inv.read_bytes()
entries=[line for line in inv.read_text().splitlines() if line.strip() and not line.lstrip().startswith('#')]
manifest={'inventory_path':str(inv),'inventory_size_bytes':len(inv_bytes),'inventory_sha256':hashlib.sha256(inv_bytes).hexdigest(),'inventory_entry_count':len(entries),'verified_inventory_file_count':report['bundle']['verified_inventory_file_count'],'bundle_sha256':report['bundle']['bundle_sha256'],'bundle_size_bytes':report['bundle']['bundle_size_bytes'],'verified':len(entries)==report['bundle']['verified_inventory_file_count']}
assert manifest['verified']
inv_evidence=EVIDENCE/f'inventory_integrity_{STAMP}.json'
assert not inv_evidence.exists()
with inv_evidence.open('x') as f: json.dump(manifest,f,indent=2,sort_keys=True); f.write('\n')
env=report['environment']; plan=report['plan_audit']
log=(EVIDENCE/f'pytest_{STAMP}.txt').read_text()
pytest_line=next((line.strip() for line in log.splitlines() if 'passed' in line), 'summary line not found')
print('STUDY_V2_PREFLIGHT_RECEIPT')
print('VERSION_DIR',VERSIONDIR)
print('SOURCE_DIR',VERSIONDIR/'src')
print('ROOT',ROOT)
print('DATA_ROOT',plan['trainval_data_root'],'train/val manifest files exist',plan['trainval_manifest_files_exist'])
print('OUTPUT_ROOT',plan['output_root'],'created',not report['training_output_root_not_created'])
print('BUNDLE_SIZE_BYTES',manifest['bundle_size_bytes'],'BUNDLE_SHA256',manifest['bundle_sha256'])
print('INVENTORY_SHA256',manifest['inventory_sha256'],'VERIFIED_FILES',manifest['verified_inventory_file_count'])
print('ENV',env['studio_kernel_torch'],'CUDA',env['studio_kernel_cuda_build'],'GPU',env['studio_kernel_gpu'])
print('SUBPROCESS_CWD',env['pytest_cwd'],'PYTHONPATH',env['pytest_pythonpath'],'IMPORT_PATH_OK',env['version_import_path_ok'])
print('PYTEST',pytest_line,'returncode',env['pytest_returncode'],'duration_s',env['pytest_duration_s'])
print('PLANNED_GRID',plan['plan_groups'],'run_counts',plan['planned_run_counts'],'total',plan['total_planned_runs'],'launched',not report['no_training_launched'])
print('COPIED_NOTEBOOK',plan['plan_copy'])
print('INVENTORY_EVIDENCE',inv_evidence)
print('EVIDENCE_FILES',sorted(p.name for p in EVIDENCE.iterdir()))

STUDY_V2_PREFLIGHT_RECEIPT
VERSION_DIR /home/sagemaker-user/brain-mri-segmentation/versions/study_v2
SOURCE_DIR /home/sagemaker-user/brain-mri-segmentation/versions/study_v2/src
ROOT /home/sagemaker-user/brain-mri-segmentation
DATA_ROOT /home/sagemaker-user/brain-mri-segmentation/data train/val manifest files exist {'train_manifest_exists': True, 'val_manifest_exists': True}
OUTPUT_ROOT /home/sagemaker-user/brain-mri-segmentation/runs/study_v2 created False
BUNDLE_SIZE_BYTES 30965 BUNDLE_SHA256 8e3577f43ebcd8d39b14912580b8806c4940c717e60b02328eca503f6e017f7d
INVENTORY_SHA256 b570ab1129eb72e86d90e63c87c40a1e5f788e3951f35ab56f0fa366c0cc65f9 VERIFIED_FILES 33
ENV 2.8.0 CUDA 12.9 GPU NVIDIA L40S
SUBPROCESS_CWD /home/sagemaker-user/brain-mri-segmentation/versions/study_v2 PYTHONPATH /home/sagemaker-user/brain-mri-segmentation/versions/study_v2/src IMPORT_PATH_OK True
PYTEST 28 passed in 2.30s returncode 0 duration_s 4.724703
PLANNED_GRID ['# MAIN: bce_baseline vs bce_dice, seeds 0,1,2 (6 ru

In [9]:
import datetime, json, re, subprocess
from pathlib import Path
from zoneinfo import ZoneInfo
ROOT=Path('/home/sagemaker-user/brain-mri-segmentation')
OUT=ROOT/'runs/study_v2'
E=ROOT/'evidence/study_v2/async_study_v2_20261108T105406343233Z'
now=datetime.datetime.now(datetime.timezone.utc)
def pick(d,*keys):
    return next((d[k] for k in keys if isinstance(d,dict) and k in d),None)
dp=E/'driver_status.json'
drv=json.loads(dp.read_text()) if dp.exists() else {}
rows=drv.get('runs',[]) if isinstance(drv,dict) else []
metas={}
for p in sorted(OUT.glob('*/run_meta.json')):
    try: metas[p.parent.name]=json.loads(p.read_text())
    except Exception as ex: metas[p.parent.name]={'_read_error':str(ex)}
def rowname(r): return pick(r,'name','run_name','run_id','id')
def rowstatus(r): return str(pick(r,'status','run_status','state') or '').lower()
completed=[r for r in rows if rowstatus(r) in ('completed','complete','succeeded','success','done')]
active=[r for r in rows if rowstatus(r) in ('running','starting','active')]
valid_names=[str(rowname(r)) for r in completed if str(rowname(r)) in metas]
selected=active[-1] if active else None
name=str(rowname(selected)) if selected and rowname(selected) else None
stdout_dir=E/'stdout'
log=stdout_dir/(name+'.log') if name else None
if (not log or not log.exists()) and stdout_dir.exists():
    candidates=list(stdout_dir.glob('*.log'))
    log=max(candidates,key=lambda p:p.stat().st_mtime) if candidates else None
latest_epoch=None
if log and log.exists():
    text=log.read_text(errors='replace')[-500000:]
    epochs=[]
    for line in text.splitlines():
        lower=line.lower()
        if 'epoch' in lower:
            nums=re.findall('[0-9]+',lower.split('epoch',1)[1])
            if nums: epochs.append(int(nums[0]))
    if epochs: latest_epoch=epochs[-1]
ps=[]
pid_values=[378891]
if selected:
    p=pick(selected,'pid','process_id','trainer_pid')
    if p is not None:
        try: pid_values.append(int(p))
        except Exception: pass
for pid in dict.fromkeys(pid_values):
    q=subprocess.run(['ps','-p',str(pid),'-o','pid=,etime=,stat=,args='],capture_output=True,text=True)
    ps.append({'pid':pid,'alive':bool(q.stdout.strip()),'line':q.stdout.strip()})
summary=[]
for r in rows:
    n=rowname(r); m=metas.get(str(n),{}) if n is not None else {}
    summary.append({'name':n,'status':rowstatus(r),'pid':pick(r,'pid','process_id','trainer_pid'),'returncode':pick(r,'returncode','exit_code','rc'),'run_meta_present':str(n) in metas if n is not None else False,'meta_fields':{k:m[k] for k in ('status','best_epoch','best_val_dice','val_dice','selected_threshold','threshold','epochs_completed','completed_utc','updated_utc') if isinstance(m,dict) and k in m}})
errs=[]
for p in ([E/'driver.log']+(list(stdout_dir.glob('*.log')) if stdout_dir.exists() else [])):
    if p.exists():
        for line in p.read_text(errors='replace').splitlines()[-1000:]:
            if any(word in line.lower() for word in ('traceback','error','exception','failed')): errs.append({'file':p.name,'line':line[-400:]})
result={'utc':now.isoformat(),'america_new_york':now.astimezone(ZoneInfo('America/New_York')).isoformat(),'driver_pid':pick(drv,'driver_pid') or 378891,'driver_status':pick(drv,'driver_status','status'),'driver_alive':next((x['alive'] for x in ps if x['pid']==378891),False),'driver_status_updated':pick(drv,'updated_utc','updated_at'),'run_rows':summary,'run_meta_file_count':len(metas),'completed_driver_count':len(completed),'completed_with_run_meta_count':len(valid_names),'completed_with_run_meta_names':valid_names,'active_run':name,'active_status':rowstatus(selected) if selected else None,'active_run_log':str(log) if log else None,'active_latest_epoch_from_log':latest_epoch,'processes':ps,'errors_found_count':len(errs),'recent_error_lines':errs[-12:]}
print('STATUS_JSON_START')
print(json.dumps(result,indent=2,default=str))
print('STATUS_JSON_END')

STATUS_JSON_START
{
  "utc": "2026-11-08T12:44:57.477201+00:00",
  "america_new_york": "2026-11-08T07:44:57.477201-05:00",
  "driver_pid": 378891,
  "driver_status": "running",
  "driver_alive": true,
  "driver_status_updated": "2026-11-08T12:33:02.845970+00:00",
  "run_rows": [
    {
      "name": "bce_baseline_s0",
      "status": "completed",
      "pid": 378892,
      "returncode": 0,
      "run_meta_present": true,
      "meta_fields": {
        "best_epoch": 47,
        "selected_threshold": 0.55
      }
    },
    {
      "name": "bce_baseline_s1",
      "status": "completed",
      "pid": 722925,
      "returncode": 0,
      "run_meta_present": true,
      "meta_fields": {
        "best_epoch": 45,
        "selected_threshold": 0.45
      }
    },
    {
      "name": "bce_baseline_s2",
      "status": "completed",
      "pid": 1061189,
      "returncode": 0,
      "run_meta_present": true,
      "meta_fields": {
        "best_epoch": 47,
        "selected_threshold": 0.55
     

In [10]:
import os,sys,json,datetime,subprocess
from pathlib import Path
ROOT=Path('/home/sagemaker-user/brain-mri-segmentation'); BR=ROOT/'evidence/study_v2/backups'
old=sorted(BR.glob('*/backup_receipt.json'),key=lambda p:p.stat().st_mtime,reverse=True)
if not old: raise RuntimeError('Prior backup receipt not found')
prior=json.loads(old[0].read_text()); bucket=prior['s3_bucket']; stamp=datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'); B=BR/stamp; B.mkdir(parents=True,exist_ok=False)
runner=B/'backup_runner.py'; log=B/'backup_runner.log'
source = """import os,sys,json,hashlib,tarfile,shutil,datetime,subprocess,traceback
from pathlib import Path
import boto3
from boto3.s3.transfer import TransferConfig
R=Path('/home/sagemaker-user/brain-mri-segmentation'); O=R/'runs/study_v2'; E=R/'evidence/study_v2/async_study_v2_20261108T105406343233Z'; B=Path(sys.argv[1]); S=B/'backup_status.json'; L=B/'completed_results_snapshot.json'; M=B/'snapshot_manifest.json'; A=B/'brain-mri-segmentation-work.tar.gz'; RUN=B/'backup_runner.py'; U=datetime.timezone.utc
start=datetime.datetime.now(U).isoformat(); state={'created_utc':start,'stage':'starting'}
def now(): return datetime.datetime.now(U).isoformat()
def put(p,x):
 q=p.with_suffix(p.suffix+'.tmp'); q.write_text(json.dumps(x,indent=2,sort_keys=True,default=str)); q.replace(p)
def step(x,**kw): state.update(stage=x,updated_utc=now(),**kw); put(S,state); print(x,json.dumps(kw,sort_keys=True,default=str),flush=True)
def sha(p):
 z=hashlib.sha256()
 with open(p,'rb') as f:
  for b in iter(lambda:f.read(1048576),b''): z.update(b)
 return z.hexdigest()
def stat(p):
 q=p.stat(); return {'size_bytes':q.st_size,'mtime_ns':q.st_mtime_ns,'mtime_utc':datetime.datetime.fromtimestamp(q.st_mtime,U).isoformat()}
def pick(d,*ks): return next((d[k] for k in ks if isinstance(d,dict) and k in d),None)
def ds():
 try: return json.loads((E/'driver_status.json').read_text())
 except Exception as e: return {'_error':str(e)}
def rows(d): return d.get('runs',[]) if isinstance(d,dict) else []
def nm(r): return str(pick(r,'name','run_name','run_id','id') or '')
def st(r): return str(pick(r,'status','run_status','state') or '').lower()
def rc(r): return pick(r,'returncode','exit_code','rc')
def done(d): return [r for r in rows(d) if st(r) in ('completed','complete','succeeded','success','done') and rc(r)==0]
def act(d):
 a=[r for r in rows(d) if st(r) in ('running','starting','active')]; r=a[-1] if a else None
 return {'name':nm(r),'status':st(r),'pid':pick(r,'pid','process_id','trainer_pid')} if r else None
EXD={'.git','.cache','cache','caches','__pycache__','.pytest_cache','.mypy_cache','.ipynb_checkpoints','node_modules','venv','.venv','.env','data','downloads','.downloads','credentials','.aws','.ssh','secrets','backups'}
IM={'.tif','.tiff','.nii','.dcm','.nrrd','.mha','.mhd','.npy','.npz','.h5','.hdf5','.png','.jpg','.jpeg','.bmp'}
def exclude(p):
 r=p.relative_to(R); parts=[x.lower() for x in r.parts]; s='/'.join(parts); n=p.name.lower()
 if any(x in EXD for x in parts): return 'excluded_directory'
 if any(x in s for x in ('test_manifest','test-manifest','testmanifest')): return 'test_manifest_not_read'
 if 'holdout' in s: return 'holdout_not_read'
 if n.startswith('.env') or any(x in n for x in ('credential','secret','token')) or n in ('id_rsa','id_ed25519') or p.suffix.lower() in ('.pem','.key'): return 'credential_secret'
 if n.endswith('.nii.gz') or p.suffix.lower() in IM: return 'image_or_array_content'
 if p.is_symlink(): return 'symlink'
 return None
def rec(p,k):
 a=stat(p); x=sha(p); b=stat(p); y=sha(p)
 return {'path':p.relative_to(R).as_posix(),'kind':k,'size_bytes':b['size_bytes'],'mtime_utc':b['mtime_utc'],'sha256':y,'stable':a==b and x==y,'stat_before':a if a!=b else None}
def collect(r,t):
 n=nm(r); d=O/n; m=d/'run_meta.json'
 if not n or not d.is_dir() or not m.is_file(): return None
 try: meta=json.loads(m.read_text())
 except Exception as e: meta={'parse_error':str(e)}
 fs=[]
 for p in sorted(d.rglob('*')):
  if not p.is_file() or exclude(p): continue
  k='checkpoint' if p.suffix.lower() in ('.pt','.pth','.ckpt','.safetensors') else ('config' if 'config' in p.name.lower() else ('log' if p.suffix.lower() in ('.log','.txt') else 'run_artifact'))
  x=rec(p,k)
  if k=='checkpoint' and not x['stable']: raise RuntimeError('Checkpoint not stable: '+str(p))
  fs.append(x)
 return {'name':n,'observed_completed_utc':t,'returncode':rc(r),'pid':pick(r,'pid','process_id','trainer_pid'),'run_dir':d.relative_to(R).as_posix(),'run_meta':meta,'best_epoch':pick(meta,'best_epoch','best_epoch_index'),'selected_threshold':pick(meta,'selected_threshold','threshold'),'files':fs,'checkpoint_sha256':{x['path']:x['sha256'] for x in fs if x['kind']=='checkpoint'},'config_sha256':{x['path']:x['sha256'] for x in fs if x['kind']=='config'}}
def active_ckpts(d):
 z=[]
 for r in rows(d):
  if st(r) not in ('running','starting','active'): continue
  q=O/nm(r)
  if q.is_dir():
   for p in q.rglob('*'):
    if p.is_file() and p.suffix.lower() in ('.pt','.pth','.ckpt','.safetensors'): z.append({'path':p.relative_to(R).as_posix(),'size_bytes':p.stat().st_size,'reason':'exclude_in_progress_checkpoint; not read or hashed'})
 return z
def epoch(d):
 a=act(d)
 if not a: return None
 sd=E/'stdout'; hs=list(sd.glob('*'+a['name']+'*')) if sd.exists() else []
 if not hs: return {'run':a['name'],'epoch':None}
 p=max(hs,key=lambda q:q.stat().st_mtime)
 with open(p,'rb') as f: f.seek(max(0,p.stat().st_size-250000)); txt=f.read().decode('utf8','replace')
 v=[]
 for line in txt.splitlines():
  if 'epoch' in line.lower():
   for token in line.lower().split('epoch',1)[1].replace(':',' ').replace('=',' ').replace('/',' ').split():
    if token.isdigit(): v.append(int(token)); break
 return {'run':a['name'],'epoch':v[-1] if v else None,'log':p.relative_to(R).as_posix(),'stat':stat(p)}
try:
 d0=ds(); recs={}; t=now()
 for r in done(d0):
  x=collect(r,t)
  if x: recs[x['name']]=x
 paths=[]; excluded=[]; er=(E.relative_to(R).as_posix()+'/')
 for base,dirs,files in os.walk(R):
  bp=Path(base); keep=[]
  for dn in dirs:
   p=bp/dn; rel=p.relative_to(R).as_posix(); why=exclude(p)
   if rel=='runs/study_v2': why='only_validated_completed_runs_included'
   if why: excluded.append({'path':rel,'reason':why})
   else: keep.append(dn)
  dirs[:]=keep
  for fn in files:
   p=bp/fn
   if p.is_file():
    why=exclude(p)
    if why: excluded.append({'path':p.relative_to(R).as_posix(),'reason':why})
    else: paths.append(p)
 g=[]
 for p in sorted(paths):
  rel=p.relative_to(R).as_posix(); live=rel.startswith(er) and (p.name in ('driver.log','driver_status.json','driver_events.jsonl') or '/stdout/' in rel)
  g.append(rec(p,'live_log_or_status' if live else 'project_file'))
 d1=ds(); before_upload=[]
 for r in done(d1):
  n=nm(r)
  if n and n not in recs:
   x=collect(r,now())
   if x: recs[n]=x; before_upload.append(n)
 light={'schema_version':1,'capture_started_utc':start,'snapshot_cutoff_utc':now(),'project_root':str(R),'driver_pid':378891,'driver_status':d0.get('driver_status'),'driver_status_updated_utc':d0.get('updated_utc'),'driver_status_at_upload':d1.get('driver_status'),'driver_status_updated_at_upload_utc':d1.get('updated_utc'),'planned_runs':14,'completed_run_count':len(recs),'completed_runs':list(recs.values()),'active_run':act(d1),'active_latest_epoch':epoch(d1),'active_checkpoints_excluded':active_ckpts(d1),'excluded':['data','downloads','caches','credentials/secrets','test manifests (not opened)','holdout paths/content (not opened)','medical images/arrays','all backups directories'],'notes':['Only driver-completed returncode 0 runs with run_meta are listed.','Checkpoint hashes are checked before and after archive copy.','Live logs are timestamped best-effort copies, not an atomic freeze.']}
 L.write_text(json.dumps(light,indent=2,sort_keys=True,default=str)); lsha=sha(L)
 old=sorted((R/'evidence/study_v2/backups').glob('*/backup_receipt.json'),key=lambda p:p.stat().st_mtime,reverse=True)[0]; prev=json.loads(old.read_text()); bucket=prev['s3_bucket']; prefix='/'.join(prev.get('s3_key','studio/backups/x/y').split('/')[:2]); s3=boto3.client('s3'); lk=prefix+'/'+B.name+'/completed_results_snapshot.json'
 s3.put_object(Bucket=bucket,Key=lk,Body=L.read_bytes(),ContentType='application/json',Metadata={'sha256':lsha}); lh=s3.head_object(Bucket=bucket,Key=lk); lb=s3.get_object(Bucket=bucket,Key=lk)['Body']; z=hashlib.sha256(); nbytes=0
 for c in lb.iter_chunks(chunk_size=1048576):
  if c: z.update(c); nbytes+=len(c)
 lb.close(); lrb=z.hexdigest()
 if lh['ContentLength']!=L.stat().st_size or lh.get('Metadata',{}).get('sha256')!=lsha or lrb!=lsha: raise RuntimeError('Light JSON readback failed')
 raw=sum(x['size_bytes'] for x in g)+sum(x['size_bytes'] for r in recs.values() for x in r['files']); free=shutil.disk_usage(R).free
 if free<int(raw*.85)+100*1024*1024: raise RuntimeError('Not enough free disk: '+str(free)+' raw='+str(raw))
 ak=prefix+'/'+B.name+'/brain-mri-segmentation-work.tar.gz'; included=list(g); livechecks=[]; late=[]
 def add(tf,p,x):
  a=stat(p); hh=sha(p) if x['kind'] in ('checkpoint','live_log_or_status') else x['sha256']
  if x['kind']=='checkpoint' and (hh!=x['sha256'] or a['size_bytes']!=x['size_bytes']): raise RuntimeError('Checkpoint changed before copy '+str(p))
  tf.add(str(p),arcname=x['path'],recursive=False); b=stat(p)
  if x['kind']=='checkpoint' and (sha(p)!=hh or a!=b): raise RuntimeError('Checkpoint changed during copy '+str(p))
  if x['kind']=='live_log_or_status': livechecks.append({'path':x['path'],'before':a,'after':b,'sha256_before':hh,'sha256_after':sha(p)})
 step('building_archive',project_files=len(g),completed_runs=len(recs),free_bytes=free)
 with tarfile.open(A,'w:gz',compresslevel=1) as tf:
  for x in g: add(tf,R/x['path'],x)
  for r in recs.values():
   for x in r['files']: add(tf,R/x['path'],x)
  dm=ds(); late_time=now()
  for r in done(dm):
   n=nm(r)
   if n and n not in recs:
    x=collect(r,late_time)
    if x:
     recs[n]=x; late.append({'name':n,'observed_completed_utc':late_time,'included_in_archive':True})
     for f in x['files']: add(tf,R/f['path'],f)
  for p,k in ((L,'completed_results_snapshot'),(runner,'backup_runner')):
   x=rec(p,k); x['path']=p.relative_to(R).as_posix(); included.append(x); tf.add(str(p),arcname=x['path'],recursive=False)
  de=ds(); cutoff=now(); pending=sorted(nm(r) for r in done(de) if nm(r) not in recs)
  manifest={'schema_version':1,'created_utc':now(),'capture_started_utc':start,'archive_cutoff_utc':cutoff,'project_root':str(R),'completed_run_count':len(recs),'completed_runs':list(recs.values()),'completed_discovered_during_capture':late,'completed_after_cutoff_not_included':pending,'active_run_at_start':act(d0),'active_run_at_cutoff':act(de),'active_checkpoints_excluded_start':active_ckpts(d0),'active_checkpoints_excluded_cutoff':active_ckpts(de),'files':included,'excluded_paths':excluded,'live_log_copy_checks':livechecks,'self_hash_excluded':True,'notes':['Active checkpoints were excluded without reading or hashing.','Completed checkpoints were stable-checked before/after archive copy.','Live logs/status are best-effort timestamped copies, not atomic.','gzip level 1; external receipt is not in archive and has no self-hash.']}
  M.write_text(json.dumps(manifest,indent=2,sort_keys=True,default=str)); mh=sha(M); tf.add(str(M),arcname=M.relative_to(R).as_posix(),recursive=False)
 asize=A.stat().st_size; ah=sha(A); s3.upload_file(str(A),bucket,ak,ExtraArgs={'Metadata':{'sha256':ah}},Config=TransferConfig(multipart_threshold=33554432,multipart_chunksize=16777216,max_concurrency=2,use_threads=True)); hd=s3.head_object(Bucket=bucket,Key=ak); body=s3.get_object(Bucket=bucket,Key=ak)['Body']; q=hashlib.sha256(); got=0
 for c in body.iter_chunks(chunk_size=8388608):
  if c: q.update(c); got+=len(c)
 body.close(); rb=q.hexdigest()
 if hd['ContentLength']!=asize or hd.get('Metadata',{}).get('sha256')!=ah or rb!=ah: raise RuntimeError('Archive readback failed')
 df=ds(); receipt={'status':'uploaded_verified','created_utc':now(),'capture_started_utc':start,'archive_cutoff_utc':cutoff,'s3_bucket':bucket,'s3_key':ak,'s3_uri':'s3://'+bucket+'/'+ak,'archive_size_bytes':asize,'archive_sha256':ah,'head_content_length':hd['ContentLength'],'head_metadata_sha256':hd.get('Metadata',{}).get('sha256'),'readback_bytes':got,'readback_sha256':rb,'completed_results_snapshot_key':lk,'completed_results_snapshot_uri':'s3://'+bucket+'/'+lk,'completed_results_snapshot_size_bytes':L.stat().st_size,'completed_results_snapshot_sha256':lsha,'completed_results_snapshot_readback_sha256':lrb,'snapshot_manifest_local':str(M),'snapshot_manifest_sha256':mh,'archive_local_path':str(A),'backup_directory':str(B),'completed_run_count':len(recs),'completed_runs':sorted(recs),'completed_during_capture':late,'completed_after_cutoff_not_included':pending,'driver_status_start':d0.get('driver_status'),'driver_status_updated_start':d0.get('updated_utc'),'driver_status_end':df.get('driver_status'),'driver_status_updated_end':df.get('updated_utc'),'active_run_end':act(df),'active_checkpoints_excluded_end':active_ckpts(df),'excluded_paths':excluded,'live_log_copy_checks':livechecks,'notes':['No data, holdout, test manifests, credentials, caches or active checkpoint included.','Prior S3 snapshots not overwritten; training and Studio left running.']}
 put(B/'backup_receipt.json',receipt); state.update(stage='uploaded_verified',updated_utc=now(),receipt=receipt); put(S,state); print('BACKUP_RECEIPT_JSON_START'); print(json.dumps(receipt,indent=2,sort_keys=True,default=str)); print('BACKUP_RECEIPT_JSON_END',flush=True)
except Exception as e:
 state.update(stage='failed',updated_utc=now(),error=str(e),traceback=traceback.format_exc()); put(S,state); print('BACKUP_FAILED '+json.dumps(state,sort_keys=True),flush=True); raise
"""
runner.write_text(source); os.chmod(runner,0o600); f=open(log,'ab',buffering=0)
p=subprocess.Popen([sys.executable,'-u',str(runner),str(B)],cwd=str(ROOT),stdin=subprocess.DEVNULL,stdout=f,stderr=subprocess.STDOUT,start_new_session=True,close_fds=True)
print('BACKUP_LAUNCH_JSON_START')
print(json.dumps({'launched':True,'created_utc':now.isoformat(),'backup_dir':str(B),'runner_pid':p.pid,'runner_log':str(log),'s3_bucket':bucket,'s3_prefix':'studio/backups/'+stamp},indent=2,sort_keys=True))
print('BACKUP_LAUNCH_JSON_END')

BACKUP_LAUNCH_JSON_START
{
  "backup_dir": "/home/sagemaker-user/brain-mri-segmentation/evidence/study_v2/backups/20261108T125446190780Z",
  "created_utc": "2026-11-08T12:44:57.477201+00:00",
  "launched": true,
  "runner_log": "/home/sagemaker-user/brain-mri-segmentation/evidence/study_v2/backups/20261108T125446190780Z/backup_runner.log",
  "runner_pid": 3686431,
  "s3_bucket": "ajay-mlp1-lgg-044450716277-use1",
  "s3_prefix": "studio/backups/20261108T125446190780Z"
}
BACKUP_LAUNCH_JSON_END


In [11]:
import os,signal,subprocess,json,hashlib,datetime,boto3
from pathlib import Path
R=Path('/home/sagemaker-user/brain-mri-segmentation'); E=R/'evidence/study_v2/async_study_v2_20261108T105406343233Z'
runner='/home/sagemaker-user/brain-mri-segmentation/evidence/study_v2/backups/20261108T125446190780Z/backup_runner.py'; pid=3686431
q=subprocess.run(['ps','-p',str(pid),'-o','args='],capture_output=True,text=True); args=q.stdout.strip(); action='not_running'
if args and runner in args and 'scripts/train.py' not in args:
 os.kill(pid,signal.SIGTERM); action='terminated_own_unapproved_backup_runner'
stamp=datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'); captured=datetime.datetime.now(datetime.timezone.utc).isoformat()
d=json.loads((E/'driver_status.json').read_text()); rows=d.get('runs',[]); out=[]
for r in rows:
 status=str(r.get('status',r.get('run_status',''))).lower(); rc=r.get('returncode',r.get('exit_code',r.get('rc'))); name=r.get('name',r.get('run_name',r.get('run_id')))
 if status in ('completed','complete','succeeded','success','done') and rc==0 and name:
  p=R/'runs/study_v2'/str(name)/'run_meta.json'
  out.append({'name':str(name),'returncode':rc,'run_meta_path':str(p) if p.exists() else None,'run_meta':json.loads(p.read_text()) if p.exists() else None})
obj={'captured_utc':captured,'driver_pid':d.get('driver_pid',378891),'driver_status':d.get('driver_status'),'driver_status_updated_utc':d.get('updated_utc'),'planned_runs':14,'completed_run_count':len(out),'completed_runs':out,'backup_runner_action':action}
data=json.dumps(obj,indent=2,sort_keys=True,default=str).encode(); path=R/'evidence/study_v2'/('results_snapshot_'+stamp+'.json'); path.write_bytes(data); sha=hashlib.sha256(data).hexdigest(); bucket='ajay-mlp1-lgg-044450716277-use1'; key='studio/results_snapshots/'+stamp+'.json'; s3=boto3.client('s3')
s3.put_object(Bucket=bucket,Key=key,Body=data,ContentType='application/json',Metadata={'sha256':sha}); head=s3.head_object(Bucket=bucket,Key=key); body=s3.get_object(Bucket=bucket,Key=key)['Body']; readback=body.read(); body.close(); verified=hashlib.sha256(readback).hexdigest()==sha and head['ContentLength']==len(data) and head.get('Metadata',{}).get('sha256')==sha
if not verified: raise RuntimeError('Results snapshot S3 verification failed')
print('RESULTS_'+'SNAPSHOT_VERIFIED'); print(json.dumps({'local_path':str(path),'s3_bucket':bucket,'s3_key':key,'s3_uri':'s3://'+bucket+'/'+key,'size_bytes':len(data),'sha256':sha,'readback_sha256':hashlib.sha256(readback).hexdigest(),'completed_run_count':len(out),'completed_run_names':[x['name'] for x in out],'driver_status':obj['driver_status'],'captured_utc':captured,'backup_runner_action':action},indent=2,sort_keys=True))

RESULTS_SNAPSHOT_VERIFIED
{
  "backup_runner_action": "not_running",
  "captured_utc": "2026-11-08T12:56:44.747475+00:00",
  "completed_run_count": 9,
  "completed_run_names": [
    "bce_baseline_s0",
    "bce_baseline_s1",
    "bce_baseline_s2",
    "bce_dice_s0",
    "bce_dice_s1",
    "bce_dice_s2",
    "weighted_bce_s0",
    "weighted_bce_s1",
    "weighted_bce_s2"
  ],
  "driver_status": "running",
  "local_path": "/home/sagemaker-user/brain-mri-segmentation/evidence/study_v2/results_snapshot_20261108T125644747427Z.json",
  "readback_sha256": "9a54bb156ae0f47c4ca432fcd01375d518e6e0e02820405f6f19564b9978ca37",
  "s3_bucket": "ajay-mlp1-lgg-044450716277-use1",
  "s3_key": "studio/results_snapshots/20261108T125644747427Z.json",
  "s3_uri": "s3://ajay-mlp1-lgg-044450716277-use1/studio/results_snapshots/20261108T125644747427Z.json",
  "sha256": "9a54bb156ae0f47c4ca432fcd01375d518e6e0e02820405f6f19564b9978ca37",
  "size_bytes": 194924
}


In [12]:
import json,datetime
from pathlib import Path
B=Path('/home/sagemaker-user/brain-mri-segmentation/evidence/study_v2/backups/20261108T125446190780Z')
out={'backup_dir':str(B),'files':[]}
for n in ('backup_status.json','backup_receipt.json','backup_runner.log','snapshot_manifest.json','completed_results_snapshot.json','brain-mri-segmentation-work.tar.gz','backup_runner.py'):
 p=B/n; out['files'].append({'name':n,'exists':p.exists(),'size_bytes':p.stat().st_size if p.exists() and p.is_file() else None,'mtime_local':datetime.datetime.fromtimestamp(p.stat().st_mtime).astimezone().isoformat() if p.exists() else None})
 if n in ('backup_status.json','backup_receipt.json','snapshot_manifest.json') and p.exists():
  try:
   x=json.loads(p.read_text()); out[n]=({k:x.get(k) for k in ('stage','status','created_utc','archive_cutoff_utc','s3_uri','s3_key','archive_size_bytes','archive_sha256','completed_run_count','completed_run_names','completed_runs','error','traceback','notes','excluded_paths','files') if k in x} if isinstance(x,dict) else None)
  except Exception as e: out[n+'_read_error']=str(e)
log=B/'backup_runner.log'
if log.exists(): out['log_tail']=log.read_text(errors='replace').splitlines()[-35:]
old=Path('/home/sagemaker-user/brain-mri-segmentation/evidence/study_v2/backups/20261108T111235894576Z')
out['prior_0612_runner']={'dir_exists':old.exists(),'runner_exists':(old/'backup_runner.py').exists(),'files':[p.name for p in old.iterdir()] if old.exists() else []}
print('RUNNER_'+'INSPECTION_DONE'); print(json.dumps(out,indent=2,sort_keys=True,default=str))

RUNNER_INSPECTION_DONE
{
  "backup_dir": "/home/sagemaker-user/brain-mri-segmentation/evidence/study_v2/backups/20261108T125446190780Z",
  "backup_status.json": {
    "completed_runs": 9,
    "created_utc": "2026-11-08T12:54:46.398286+00:00",
    "error": "name 'runner' is not defined",
    "stage": "failed",
    "traceback": "Traceback (most recent call last):\n  File \"/home/sagemaker-user/brain-mri-segmentation/evidence/study_v2/backups/20261108T125446190780Z/backup_runner.py\", line 139, in <module>\n    for p,k in ((L,'completed_results_snapshot'),(runner,'backup_runner')):\n                                                  ^^^^^^\nNameError: name 'runner' is not defined\n"
  },
  "files": [
    {
      "exists": true,
      "mtime_local": "2026-11-08T12:54:58.978163+00:00",
      "name": "backup_status.json",
      "size_bytes": 624
    },
    {
      "exists": false,
      "mtime_local": null,
      "name": "backup_receipt.json",
      "size_bytes": null
    },
    {
      "exis

In [13]:
import json,hashlib,subprocess
from pathlib import Path
import boto3
B=Path('/home/sagemaker-user/brain-mri-segmentation/evidence/study_v2/backups/20261108T125446190780Z'); P=Path('/home/sagemaker-user/brain-mri-segmentation/evidence/study_v2/backups/20261108T111235894576Z')
out={'custom_status':json.loads((B/'backup_status.json').read_text()),'custom_archive_exists':(B/'brain-mri-segmentation-work.tar.gz').exists(),'custom_archive_size':(B/'brain-mri-segmentation-work.tar.gz').stat().st_size if (B/'brain-mri-segmentation-work.tar.gz').exists() else None,'custom_receipt_exists':(B/'backup_receipt.json').exists(),'custom_manifest_exists':(B/'snapshot_manifest.json').exists()}
old=P/'snapshot_upload.py'; out['prior_script']={'path':str(old),'exists':old.exists()}
if old.exists():
 t=old.read_text(errors='replace'); out['prior_script'].update(size=len(t),sha256=hashlib.sha256(t.encode()).hexdigest(),lines=len(t.splitlines()),safety_lines=[f'{i+1}: {ln[:220]}' for i,ln in enumerate(t.splitlines()) if any(k in ln.lower() for k in ('strftime','timestamp','datetime','rmtree','unlink(','remove(','upload_file','put_object','backup','completed','checkpoint','run_meta','sys.argv','s3_key','tarfile','head_object'))][:45])
oldrec=json.loads((P/'backup_receipt.json').read_text()) if (P/'backup_receipt.json').exists() else {}
out['prior_receipt']={k:oldrec.get(k) for k in ('status','created_utc','s3_bucket','s3_key','archive_size_bytes','archive_sha256','completed_runs_at_start','completed_checkpoint_count')}
s3=boto3.client('s3'); bucket=oldrec.get('s3_bucket'); prefix='studio/backups/20261108T125446190780Z/'
if bucket:
 for name in ('brain-mri-segmentation-work.tar.gz','completed_results_snapshot.json'):
  try:
   h=s3.head_object(Bucket=bucket,Key=prefix+name); out['s3_'+name]={'exists':True,'size_bytes':h['ContentLength'],'metadata':h.get('Metadata',{})}
  except Exception as e: out['s3_'+name]={'exists':False,'error':str(e)[:220]}
print('EXISTING_RUNNER_INSPECTION_DONE'); print(json.dumps(out,indent=2,sort_keys=True,default=str))

EXISTING_RUNNER_INSPECTION_DONE
{
  "custom_archive_exists": true,
  "custom_archive_size": 319120677,
  "custom_manifest_exists": false,
  "custom_receipt_exists": false,
  "custom_status": {
    "completed_runs": 9,
    "created_utc": "2026-11-08T12:54:46.398286+00:00",
    "error": "name 'runner' is not defined",
    "free_bytes": 18906292224,
    "project_files": 130,
    "stage": "failed",
    "traceback": "Traceback (most recent call last):\n  File \"/home/sagemaker-user/brain-mri-segmentation/evidence/study_v2/backups/20261108T125446190780Z/backup_runner.py\", line 139, in <module>\n    for p,k in ((L,'completed_results_snapshot'),(runner,'backup_runner')):\n                                                  ^^^^^^\nNameError: name 'runner' is not defined\n",
    "updated_utc": "2026-11-08T12:54:58.985311+00:00"
  },
  "prior_receipt": {
    "archive_sha256": "6b03029d6cc3069909474b387cd072cebd1f7ee972df0e3a2741b78aad953362",
    "archive_size_bytes": 87753529,
    "completed_che

In [14]:
from pathlib import Path
p=Path('/home/sagemaker-user/brain-mri-segmentation/evidence/study_v2/backups/20261108T111235894576Z/snapshot_upload.py'); lines=p.read_text().splitlines(); picks=list(range(0,18))+list(range(35,56))+list(range(90,115)); print('PROVEN_RUNNER_LINES'); print('
'.join(f'{i+1}: {lines[i][:420]}' for i in picks if i<len(lines)))

SyntaxError: unterminated string literal (detected at line 2) (963512396.py, line 2)

In [15]:
from pathlib import Path
p=Path('/home/sagemaker-user/brain-mri-segmentation/evidence/study_v2/backups/20261108T111235894576Z/snapshot_upload.py'); lines=p.read_text().splitlines(); picks=list(range(0,18))+list(range(35,56))+list(range(90,115)); print('PROVEN_RUNNER_LINES')
for i in picks:
 if i<len(lines): print(str(i+1)+': '+lines[i][:420])

PROVEN_RUNNER_LINES
1: import os,sys,json,hashlib,tarfile,shutil,datetime,time,base64,re,traceback
2: from pathlib import Path
3: import boto3
4: from botocore.exceptions import ClientError
5: ROOT=Path('/home/sagemaker-user/brain-mri-segmentation');BACKUP=Path(sys.argv[1]);STAMP=sys.argv[2];DRIVER=ROOT/'evidence/study_v2/async_study_v2_20261108T105406343233Z';STATUS=DRIVER/'driver_status.json';MF=DRIVER/'launch_snapshot.json';BUCKET='ajay-mlp1-lgg-044450716277-use1';KEY='studio/backups/'+STAMP+'/brain-mri-segmentation-work.tar.gz';ARCHIVE=BACKUP/'brain-mri-segmentation-work.tar.gz';STAGE=BACKUP/'staging';
6: def utc():return datetime.datetime.now(datetime.timezone.utc).isoformat()
7: def statrec(p):
8:  s=p.stat();return {'size':s.st_size,'mtime_ns':s.st_mtime_ns,'mtime_utc':datetime.datetime.fromtimestamp(s.st_mtime,datetime.timezone.utc).isoformat()}
9: def sha(p):
10:  h=hashlib.sha256()
11:  with open(p,'rb') as f:
12:   for b in iter(lambda:f.read(1024*1024),b''):h.update(b)
13: 

In [16]:
from pathlib import Path
p=Path('/home/sagemaker-user/brain-mri-segmentation/evidence/study_v2/backups/20261108T111235894576Z/snapshot_upload.py'); a=p.read_text().splitlines()
for i in list(range(0,15))+list(range(54,91)):
 if i<len(a): print(str(i+1)+': '+a[i][:360])

1: import os,sys,json,hashlib,tarfile,shutil,datetime,time,base64,re,traceback
2: from pathlib import Path
3: import boto3
4: from botocore.exceptions import ClientError
5: ROOT=Path('/home/sagemaker-user/brain-mri-segmentation');BACKUP=Path(sys.argv[1]);STAMP=sys.argv[2];DRIVER=ROOT/'evidence/study_v2/async_study_v2_20261108T105406343233Z';STATUS=DRIVER/'driver_status.json';MF=DRIVER/'launch_snapshot.json';BUCKET='ajay-mlp1-lgg-044450716277-use1';KEY='studio/backups/'+STAMP+'/brain-mri-segmentation-work.tar.gz';ARCHIVE=BACKUP/
6: def utc():return datetime.datetime.now(datetime.timezone.utc).isoformat()
7: def statrec(p):
8:  s=p.stat();return {'size':s.st_size,'mtime_ns':s.st_mtime_ns,'mtime_utc':datetime.datetime.fromtimestamp(s.st_mtime,datetime.timezone.utc).isoformat()}
9: def sha(p):
10:  h=hashlib.sha256()
11:  with open(p,'rb') as f:
12:   for b in iter(lambda:f.read(1024*1024),b''):h.update(b)
13:  return h.hexdigest()
14: def jwrite(p,x):p.write_text(json.dumps(x,indent=2,sor

In [17]:
from pathlib import Path
p=Path('/home/sagemaker-user/brain-mri-segmentation/evidence/study_v2/backups/20261108T111235894576Z/snapshot_upload.py'); a=p.read_text().splitlines()
for i in [4,14,15,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114]:
 if i<len(a): print(str(i+1)+': '+a[i][:950])

5: ROOT=Path('/home/sagemaker-user/brain-mri-segmentation');BACKUP=Path(sys.argv[1]);STAMP=sys.argv[2];DRIVER=ROOT/'evidence/study_v2/async_study_v2_20261108T105406343233Z';STATUS=DRIVER/'driver_status.json';MF=DRIVER/'launch_snapshot.json';BUCKET='ajay-mlp1-lgg-044450716277-use1';KEY='studio/backups/'+STAMP+'/brain-mri-segmentation-work.tar.gz';ARCHIVE=BACKUP/'brain-mri-segmentation-work.tar.gz';STAGE=BACKUP/'staging';NL=chr(10)
15: status0=json.load(open(STATUS)) if STATUS.exists() else {};snap=json.load(open(MF)) if MF.exists() else {};rows=status0.get('runs',[]);active=[x for x in rows if x.get('status')=='running'];completed=[x for x in rows if x.get('status')=='completed']
16: def ep_for(row):
19:  try:
20:   text=p.read_text(errors='replace');m=list(re.finditer(r'"epoch"\s*:\s*(\d+)',text));return int(m[-1].group(1)) if m else None
21:  except Exception:return None
22: run_summary=[];first_val=None
23: for x in rows:
24:  q={'name':x.get('name'),'status':x.get('status'),'pid':x.

In [18]:
import json,hashlib,datetime,subprocess,sys,os
from pathlib import Path
R=Path('/home/sagemaker-user/brain-mri-segmentation'); base=R/'evidence/study_v2/backups'; script=base/'20261108T111235894576Z/snapshot_upload.py'; stamp=datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'); b=base/stamp; b.mkdir(parents=True,exist_ok=False)
sh=hashlib.sha256(script.read_bytes()).hexdigest(); start={'started_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),'stamp':stamp,'runner':str(script),'runner_sha256':sh,'s3_bucket':'ajay-mlp1-lgg-044450716277-use1','s3_key':'studio/backups/'+stamp+'/brain-mri-segmentation-work.tar.gz'}
(b/'snapshot_started.json').write_text(json.dumps(start,indent=2,sort_keys=True))
f=open(b/'snapshot_upload.log','ab',buffering=0); p=subprocess.Popen([sys.executable,str(script),str(b),stamp],cwd=str(R),stdin=subprocess.DEVNULL,stdout=f,stderr=subprocess.STDOUT,start_new_session=True,close_fds=True)
start['runner_pid']=p.pid; (b/'snapshot_started.json').write_text(json.dumps(start,indent=2,sort_keys=True)); print('PROVEN_BACKUP_LAUNCHED'); print(json.dumps(start,indent=2,sort_keys=True))

PROVEN_BACKUP_LAUNCHED
{
  "runner": "/home/sagemaker-user/brain-mri-segmentation/evidence/study_v2/backups/20261108T111235894576Z/snapshot_upload.py",
  "runner_pid": 3868720,
  "runner_sha256": "8eacab0b3a8d6ef5dc413397425b45b916728d82496aef49b29924186ae02456",
  "s3_bucket": "ajay-mlp1-lgg-044450716277-use1",
  "s3_key": "studio/backups/20261108T130128727602Z/brain-mri-segmentation-work.tar.gz",
  "stamp": "20261108T130128727602Z",
  "started_utc": "2026-11-08T13:01:28.728418+00:00"
}


In [19]:
import json,subprocess
from pathlib import Path
B=Path('/home/sagemaker-user/brain-mri-segmentation/evidence/study_v2/backups/20261108T130128727602Z'); pid='3868720'; q=subprocess.run(['ps','-p',pid,'-o','pid=,etime=,stat=,args='],capture_output=True,text=True); o={'pid':pid,'process_line':q.stdout.strip() or None,'files':[{'name':p.name,'size':p.stat().st_size} for p in sorted(B.iterdir()) if p.is_file()]}; log=B/'snapshot_upload.log'; o['log_tail']=log.read_text(errors='replace').splitlines()[-25:] if log.exists() else []
r=B/'backup_receipt.json'; o['receipt']=json.loads(r.read_text()) if r.exists() else None; print('PROVEN_RUNNER_STATUS'); print(json.dumps(o,indent=2,sort_keys=True,default=str))

PROVEN_RUNNER_STATUS
{
  "files": [
    {
      "name": "backup_receipt.json",
      "size": 3033
    },
    {
      "name": "brain-mri-segmentation-work.tar.gz",
      "size": 348121788
    },
    {
      "name": "snapshot_manifest.json",
      "size": 173512
    },
    {
      "name": "snapshot_started.json",
      "size": 469
    },
    {
      "name": "snapshot_upload.log",
      "size": 2818
    }
  ],
  "log_tail": [
    "BACKUP_DONE {\"active_runs_at_start\": [{\"epoch_latest\": 15, \"name\": \"bce_dice_posonly_s1\", \"pid\": 3767289}], \"archive_local_path\": \"/home/sagemaker-user/brain-mri-segmentation/evidence/study_v2/backups/20261108T130128727602Z/brain-mri-segmentation-work.tar.gz\", \"archive_sha256\": \"089d2f4511000b7f3baf26600e7bfc9594bcc707b8370c86edd4c0e905baa34f\", \"archive_size_bytes\": 348121788, \"backup_directory\": \"/home/sagemaker-user/brain-mri-segmentation/evidence/study_v2/backups/20261108T130128727602Z\", \"checksum_matches\": true, \"completed_checkpoi

In [20]:
import json,hashlib,base64,tarfile,boto3
from pathlib import Path
B=Path('/home/sagemaker-user/brain-mri-segmentation/evidence/study_v2/backups/20261108T130128727602Z'); A=B/'brain-mri-segmentation-work.tar.gz'; M=B/'snapshot_manifest.json'; R=json.loads((B/'backup_receipt.json').read_text()); m=json.loads(M.read_text())
def sha_file(p):
 h=hashlib.sha256()
 with open(p,'rb') as f:
  for x in iter(lambda:f.read(1048576),b''): h.update(x)
 return h.hexdigest()
records=m.get('included',m.get('included_files',m.get('files',[])))
expected={'brain-mri-segmentation/'+x['path']:x for x in records if isinstance(x,dict) and x.get('path')}
mp='brain-mri-segmentation/evidence/study_v2/backup_snapshot_manifest.json'
with tarfile.open(A,'r:gz') as tf:
 members={x.name:x for x in tf.getmembers()}; names=set(members); missing=sorted(set(expected)-names); extra=sorted(names-set(expected)-{mp}); verified=0; mismatches=[]
 for n,x in expected.items():
  if n not in members: continue
  f=tf.extractfile(members[n]); h=hashlib.sha256()
  for b in iter(lambda:f.read(1048576),b''): h.update(b)
  f.close(); actual=h.hexdigest()
  if actual!=x.get('sha256'): mismatches.append({'path':n,'expected':x.get('sha256'),'actual':actual})
  else: verified+=1
 mf=tf.extractfile(members[mp]) if mp in members else None; member_sha=hashlib.sha256(mf.read()).hexdigest() if mf else None
local={'size_bytes':A.stat().st_size,'sha256':sha_file(A)}; s3=boto3.client('s3'); head=s3.head_object(Bucket=R['s3_bucket'],Key=R['s3_key'],ChecksumMode='ENABLED'); b64=base64.b64encode(bytes.fromhex(R['archive_sha256'])).decode()
result={'receipt_status':R['status'],'capture_started_utc':json.loads((B/'snapshot_started.json').read_text())['started_utc'],'receipt_created_utc':R['created_utc'],'manifest_created_utc':m.get('created_utc'),'manifest_keys':sorted(m.keys()),'completed_runs_at_start':R['completed_runs_at_start'],'active_runs_at_start':m.get('active_runs_at_start'),'driver_status_start_end':[R['driver_status_at_start'],R['driver_status_at_end']],'included_file_count':R['included_file_count'],'manifest_record_count':len(records),'archive_member_count':len(names),'missing_manifest_members':missing,'unlisted_archive_members':extra,'file_hashes_verified':verified,'file_hash_mismatches':mismatches,'manifest_member_present':mp in names,'manifest_member_sha256':member_sha,'manifest_local_sha256':R['snapshot_manifest_sha256'],'local_archive':local,'s3_key':R['s3_key'],'s3_size_bytes':head['ContentLength'],'s3_metadata_sha256':head.get('Metadata',{}).get('sha256'),'s3_checksum_matches_receipt':head.get('ChecksumSHA256')==b64,'size_matches_receipt':head['ContentLength']==R['archive_size_bytes'],'sha_matches_receipt':local['sha256']==R['archive_sha256'] and head.get('Metadata',{}).get('sha256')==local['sha256'],'excluded_directory_count':R['excluded_directory_count'],'skipped_file_count':R['skipped_file_count'],'completed_checkpoint_count':R['completed_checkpoint_count'],'notes':R['notes']}
print('ARCHIVE_MANIFEST_VERIFICATION'); print(json.dumps(result,indent=2,sort_keys=True,default=str))

ARCHIVE_MANIFEST_VERIFICATION
{
  "active_runs_at_start": [
    {
      "epoch_latest": 14,
      "name": "bce_dice_posonly_s1",
      "pid": 3767289,
      "run_dir": "/home/sagemaker-user/brain-mri-segmentation/runs/study_v2/bce_dice_posonly_s1"
    }
  ],
  "archive_member_count": 175,
  "capture_started_utc": "2026-11-08T13:01:28.728418+00:00",
  "completed_checkpoint_count": 12,
  "completed_runs_at_start": 10,
  "driver_status_start_end": [
    "running",
    "running"
  ],
  "excluded_directory_count": 9,
  "file_hash_mismatches": [],
  "file_hashes_verified": 174,
  "included_file_count": 174,
  "local_archive": {
    "sha256": "089d2f4511000b7f3baf26600e7bfc9594bcc707b8370c86edd4c0e905baa34f",
    "size_bytes": 348121788
  },
  "manifest_created_utc": "2026-11-08T13:01:29.774608+00:00",
  "manifest_keys": [
    "active_runs_at_start",
    "backup_stamp_utc",
    "completed_runs_at_start",
    "created_utc",
    "current_run_at_start",
    "driver_pid",
    "driver_status_at_en

In [21]:
import json
from pathlib import Path
B=Path('/home/sagemaker-user/brain-mri-segmentation/evidence/study_v2/backups/20261108T130128727602Z'); m=json.loads((B/'snapshot_manifest.json').read_text()); r=json.loads((B/'backup_receipt.json').read_text());
runs=m.get('run_summaries',[]); out={'completed_runs':[{'name':x.get('name'),'status':x.get('status'),'epoch_latest':x.get('epoch_latest'),'best_epoch':x.get('best_epoch'),'selected_threshold':x.get('selected_threshold')} for x in runs if x.get('status')=='completed'],'active_runs_at_start':m.get('active_runs_at_start'),'current_run_at_start':m.get('current_run_at_start'),'excluded_directories':m.get('excluded_directories'),'skipped_files':m.get('skipped_files'),'unstable_completed_checkpoints_excluded':m.get('unstable_completed_checkpoints_excluded'),'included_file_count':len(m.get('included_files',[])),'completed_checkpoint_count':r.get('completed_checkpoint_count'),'errors_found':r.get('errors_found')}
print('MANIFEST_SCOPE_SUMMARY'); print(json.dumps(out,indent=2,sort_keys=True,default=str))

MANIFEST_SCOPE_SUMMARY
{
  "active_runs_at_start": [
    {
      "epoch_latest": 14,
      "name": "bce_dice_posonly_s1",
      "pid": 3767289,
      "run_dir": "/home/sagemaker-user/brain-mri-segmentation/runs/study_v2/bce_dice_posonly_s1"
    }
  ],
  "completed_checkpoint_count": 12,
  "completed_runs": [
    {
      "best_epoch": 47,
      "epoch_latest": 49,
      "name": "bce_baseline_s0",
      "selected_threshold": 0.55,
      "status": "completed"
    },
    {
      "best_epoch": 45,
      "epoch_latest": 49,
      "name": "bce_baseline_s1",
      "selected_threshold": 0.45,
      "status": "completed"
    },
    {
      "best_epoch": 47,
      "epoch_latest": 49,
      "name": "bce_baseline_s2",
      "selected_threshold": 0.55,
      "status": "completed"
    },
    {
      "best_epoch": 27,
      "epoch_latest": 49,
      "name": "bce_dice_s0",
      "selected_threshold": 0.2,
      "status": "completed"
    },
    {
      "best_epoch": 48,
      "epoch_latest": 49,
      "

In [ ]:
import json,hashlib,datetime,subprocess,sys,os,boto3
from pathlib import Path
from zoneinfo import ZoneInfo
R=Path('/home/sagemaker-user/brain-mri-segmentation'); E=R/'evidence/study_v2/async_study_v2_20261108T105406343233Z'; t=datetime.datetime.now(datetime.timezone.utc); stamp=t.strftime('%Y%m%dT%H%M%S%fZ')
d=json.loads((E/'driver_status.json').read_text()); rows=d.get('runs',[])
def nm(x): return str(x.get('name',x.get('run_name',x.get('run_id',''))))
def st(x): return str(x.get('status',x.get('run_status',''))).lower()
def rc(x): return x.get('returncode',x.get('exit_code',x.get('rc')))
def sh(p):
 h=hashlib.sha256()
 with open(p,'rb') as f:
  for b in iter(lambda:f.read(1048576),b''): h.update(b)
 return h.hexdigest()
completed=[]
for x in rows:
 if st(x)=='completed' and rc(x)==0 and nm(x):
  p=R/'runs/study_v2'/nm(x)/'run_meta.json'
  if p.exists(): completed.append({'name':nm(x),'returncode':rc(x),'pid':x.get('pid'),'run_meta_path':str(p),'run_meta':json.loads(p.read_text())})
active=[x for x in rows if st(x) in ('running','starting','active')]; cur=active[-1] if active else None
log=Path(cur.get('stdout','')) if cur and cur.get('stdout') else None
if not log or not log.exists():
 hs=list((E/'stdout').glob('*'+nm(cur)+'*')) if cur else []; log=max(hs,key=lambda p:p.stat().st_mtime) if hs else None
epoch=None
if log and log.exists():
 txt=log.read_text(errors='replace').splitlines()[-1500:]
 for line in txt:
  if 'epoch' in line.lower():
   for q in line.lower().split('epoch',1)[1].replace(':',' ').replace('=',' ').replace('/',' ').split():
    if q.isdigit(): epoch=int(q); break
errors=[]
for p in [E/'driver.log']+[Path(x['stdout']) for x in rows if x.get('stdout') and Path(x['stdout']).exists()]:
 for line in p.read_text(errors='replace').splitlines()[-1500:]:
  if any(w in line.lower() for w in ('traceback','error','exception','failed')): errors.append({'file':p.name,'line':line[-300:]})
ps=[]
for pid in [d.get('driver_pid',378891)]+([cur.get('pid')] if cur and cur.get('pid') else []):
 q=subprocess.run(['ps','-p',str(pid),'-o','pid=,etime=,stat=,args='],capture_output=True,text=True); ps.append({'pid':pid,'alive':bool(q.stdout.strip()),'line':q.stdout.strip()})
prevdir=R/'evidence/study_v2/backups/20261108T130128727602Z'; prevrec=json.loads((prevdir/'backup_receipt.json').read_text()); pm=json.loads((prevdir/'snapshot_manifest.json').read_text()); expect={x['path']:x.get('sha256') for x in pm.get('included_files',[]) if x.get('category')=='completed_checkpoint'}
checks=[]
for x in completed:
 p=R/'runs/study_v2'/x['name']; choices=[p/'best.pt',p/'best.pth']
 cp=next((q for q in choices if q.is_file()),None)
 if cp:
  a=cp.stat(); h1=sh(cp); b=cp.stat(); h2=sh(cp); rel=cp.relative_to(R).as_posix(); checks.append({'run':x['name'],'path':rel,'stable_hash':a.st_size==b.st_size and a.st_mtime_ns==b.st_mtime_ns and h1==h2,'sha256':h2,'previous_manifest_match':(h2==expect[rel]) if rel in expect else None})
active_check=[]
if cur:
 p=R/'runs/study_v2'/nm(cur)
 for q in p.glob('*.pt') if p.exists() else []: active_check.append({'path':str(q),'size_bytes':q.stat().st_size,'mtime_utc':datetime.datetime.fromtimestamp(q.stat().st_mtime,datetime.timezone.utc).isoformat(),'action':'active checkpoint not copied'})
obj={'captured_utc':t.isoformat(),'captured_est':t.astimezone(ZoneInfo('America/New_York')).isoformat(),'driver_pid':d.get('driver_pid',378891),'driver_status':d.get('driver_status'),'driver_status_updated_utc':d.get('updated_utc'),'driver_alive':next((x['alive'] for x in ps if x['pid']==378891),False),'processes':ps,'planned_runs':14,'completed_run_count':len(completed),'completed_runs':completed,'active_run':{'name':nm(cur),'status':st(cur),'pid':cur.get('pid'),'latest_epoch':epoch,'stdout':str(log) if log else None} if cur else None,'errors_found_count':len(errors),'recent_errors':errors[-12:],'completed_checkpoint_hash_checks':checks,'active_checkpoints':active_check}
local=R/'evidence/study_v2'/('results_snapshot_'+stamp+'.json'); data=json.dumps(obj,indent=2,sort_keys=True,default=str).encode(); local.write_bytes(data); digest=hashlib.sha256(data).hexdigest()
backup_launch=None
if len(completed)>int(prevrec.get('completed_runs_at_start',10)):
 script=R/'evidence/study_v2/backups/20261108T111235894576Z/snapshot_upload.py'; expected_script='8eacab0b3a8d6ef5dc413397425b45b916728d82496aef49b29924186ae02456'
 if script.exists() and sh(script)==expected_script:
  bt=datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'); B=R/'evidence/study_v2/backups'/bt; B.mkdir(parents=True,exist_ok=False); start={'started_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),'stamp':bt,'runner':str(script),'runner_sha256':expected_script}; (B/'snapshot_started.json').write_text(json.dumps(start,indent=2,sort_keys=True)); f=open(B/'snapshot_upload.log','ab',buffering=0); p=subprocess.Popen([sys.executable,str(script),str(B),bt],cwd=str(R),stdin=subprocess.DEVNULL,stdout=f,stderr=subprocess.STDOUT,start_new_session=True,close_fds=True); start['runner_pid']=p.pid; (B/'snapshot_started.json').write_text(json.dumps(start,indent=2,sort_keys=True)); backup_launch={'stamp':bt,'pid':p.pid,'backup_dir':str(B),'runner_sha256':expected_script}
 else: backup_launch={'error':'proven runner missing or hash changed; not launched'}
obj['backup_launch']=backup_launch; s3=boto3.client('s3',region_name='us-east-1'); key='studio/results_snapshots/'+stamp+'.json'; s3.put_object(Bucket='ajay-mlp1-lgg-044450716277-use1',Key=key,Body=data,ContentType='application/json',Metadata={'sha256':digest}); head=s3.head_object(Bucket='ajay-mlp1-lgg-044450716277-use1',Key=key); body=s3.get_object(Bucket='ajay-mlp1-lgg-044450716277-use1',Key=key)['Body']; rb=hashlib.sha256(body.read()).hexdigest(); body.close()
if head['ContentLength']!=len(data) or head.get('Metadata',{}).get('sha256')!=digest or rb!=digest: raise RuntimeError('Status JSON S3 verification failed')
print('FRESH_STATUS_SNAPSHOT'); print(json.dumps({'local_path':str(local),'s3_uri':'s3://ajay-mlp1-lgg-044450716277-use1/'+key,'size_bytes':len(data),'sha256':digest,'readback_sha256':rb,'captured_utc':t.isoformat(),'captured_est':obj['captured_est'],'driver_status':obj['driver_status'],'driver_alive':obj['driver_alive'],'completed_run_count':len(completed),'completed_run_names':[x['name'] for x in completed],'active_run':obj['active_run'],'errors_found_count':len(errors),'checkpoint_hash_checks':checks,'active_checkpoints':active_check,'backup_launch':backup_launch},indent=2,sort_keys=True,default=str))

In [22]:
import json,hashlib,datetime,subprocess,sys,boto3,os
from pathlib import Path
from zoneinfo import ZoneInfo
R=Path('/home/sagemaker-user/brain-mri-segmentation'); E=R/'evidence/study_v2/async_study_v2_20261108T105406343233Z'; t=datetime.datetime.now(datetime.timezone.utc); stamp=t.strftime('%Y%m%dT%H%M%S%fZ')
d=json.loads((E/'driver_status.json').read_text()); rows=d.get('runs',[])
def name(x): return str(x.get('name',x.get('run_name',x.get('run_id',''))))
def status(x): return str(x.get('status',x.get('run_status',''))).lower()
def rc(x): return x.get('returncode',x.get('exit_code',x.get('rc')))
def sha(p):
 h=hashlib.sha256()
 with open(p,'rb') as f:
  for b in iter(lambda:f.read(1048576),b''): h.update(b)
 return h.hexdigest()
done=[x for x in rows if status(x)=='completed' and rc(x)==0 and (R/'runs/study_v2'/name(x)/'run_meta.json').exists()]; active=[x for x in rows if status(x) in ('running','starting','active')]; cur=active[-1] if active else None
logs=[E/'driver.log']+[Path(x['stdout']) for x in rows if x.get('stdout') and Path(x['stdout']).exists()]; errors=[]
def tail(p):
 with open(p,'rb') as f: f.seek(max(0,p.stat().st_size-150000)); return f.read().decode('utf8','replace').splitlines()
for p in logs:
 for line in tail(p)[-500:]:
  if any(w in line.lower() for w in ('traceback','error','exception','failed')): errors.append({'file':p.name,'line':line[-250:]})
log=Path(cur.get('stdout','')) if cur and cur.get('stdout') else None
if not log or not log.exists():
 hs=list((E/'stdout').glob('*'+name(cur)+'*')) if cur else []; log=max(hs,key=lambda p:p.stat().st_mtime) if hs else None
epoch=None
if log and log.exists():
 for line in tail(log)[-500:]:
  if 'epoch' in line.lower():
   for z in line.lower().split('epoch',1)[1].replace(':',' ').replace('=',' ').replace('/',' ').split():
    if z.isdigit(): epoch=int(z); break
ps=[]
for pid in [d.get('driver_pid',378891)]+([cur.get('pid')] if cur and cur.get('pid') else []):
 q=subprocess.run(['ps','-p',str(pid),'-o','pid=,etime=,stat=,args='],capture_output=True,text=True); ps.append({'pid':pid,'alive':bool(q.stdout.strip()),'line':q.stdout.strip()})
prev=R/'evidence/study_v2/backups/20261108T130128727602Z'; pm=json.loads((prev/'snapshot_manifest.json').read_text()); expect={x['path']:x.get('sha256') for x in pm.get('included_files',[]) if x.get('category')=='completed_checkpoint'}
ck=None
if done:
 p=R/'runs/study_v2'/name(done[-1])/'best.pt'
 if p.exists():
  a=p.stat(); h1=sha(p); b=p.stat(); h2=sha(p); rel=p.relative_to(R).as_posix(); ck={'path':rel,'stable_hash':a.st_size==b.st_size and a.st_mtime_ns==b.st_mtime_ns and h1==h2,'sha256':h2,'previous_manifest_match':h2==expect[rel] if rel in expect else None}
actck=[]
if cur:
 p=R/'runs/study_v2'/name(cur)/'best.pt'
 if p.exists(): actck=[{'path':str(p),'size_bytes':p.stat().st_size,'mtime_utc':datetime.datetime.fromtimestamp(p.stat().st_mtime,datetime.timezone.utc).isoformat(),'action':'active; not copied or hashed'}]
metas=[{'name':name(x),'returncode':rc(x),'run_meta':json.loads((R/'runs/study_v2'/name(x)/'run_meta.json').read_text())} for x in done]
obj={'captured_utc':t.isoformat(),'captured_est':t.astimezone(ZoneInfo('America/New_York')).isoformat(),'driver_status_snapshot':d,'driver_pid':d.get('driver_pid',378891),'driver_alive':next((x['alive'] for x in ps if x['pid']==378891),False),'processes':ps,'planned_runs':14,'completed_run_count':len(done),'completed_runs':metas,'active_run':{'name':name(cur),'status':status(cur),'pid':cur.get('pid'),'latest_epoch':epoch,'stdout':str(log) if log else None} if cur else None,'errors_found_count':len(errors),'recent_errors':errors[-10:],'latest_completed_checkpoint_hash_check':ck,'active_checkpoints':actck}
local=R/'evidence/study_v2'/('results_snapshot_'+stamp+'.json'); data=json.dumps(obj,indent=2,sort_keys=True,default=str).encode(); local.write_bytes(data)
prevrec=json.loads((prev/'backup_receipt.json').read_text()); launch=None
if len(done)>int(prevrec.get('completed_runs_at_start',10)):
 script=R/'evidence/study_v2/backups/20261108T111235894576Z/snapshot_upload.py'; expected='8eacab0b3a8d6ef5dc413397425b45b916728d82496aef49b29924186ae02456'
 if script.exists() and sha(script)==expected:
  bt=datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'); B=R/'evidence/study_v2/backups'/bt; B.mkdir(parents=True,exist_ok=False); (B/'snapshot_started.json').write_text(json.dumps({'started_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),'stamp':bt,'runner':str(script),'runner_sha256':expected},indent=2,sort_keys=True)); f=open(B/'snapshot_upload.log','ab',buffering=0); p=subprocess.Popen([sys.executable,str(script),str(B),bt],cwd=str(R),stdin=subprocess.DEVNULL,stdout=f,stderr=subprocess.STDOUT,start_new_session=True,close_fds=True); launch={'stamp':bt,'pid':p.pid,'backup_dir':str(B),'runner_sha256':expected}
 else: launch={'error':'known runner missing or hash changed; not launched'}
bucket='ajay-mlp1-lgg-044450716277-use1'; key='studio/results_snapshots/'+stamp+'.json'; digest=hashlib.sha256(data).hexdigest(); s3=boto3.client('s3',region_name='us-east-1'); s3.put_object(Bucket=bucket,Key=key,Body=data,ContentType='application/json',Metadata={'sha256':digest}); head=s3.head_object(Bucket=bucket,Key=key); body=s3.get_object(Bucket=bucket,Key=key)['Body']; rb=hashlib.sha256(body.read()).hexdigest(); body.close()
if head['ContentLength']!=len(data) or head.get('Metadata',{}).get('sha256')!=digest or rb!=digest: raise RuntimeError('status snapshot S3 verification failed')
print('STATUS_READY'); print(json.dumps({'captured_utc':t.isoformat(),'completed_run_count':len(done),'completed_run_names':[name(x) for x in done],'driver_status':d.get('driver_status'),'driver_alive':obj['driver_alive'],'active_run':obj['active_run'],'errors_found_count':len(errors),'checkpoint_check':ck,'active_checkpoints':actck,'local_path':str(local),'s3_uri':'s3://'+bucket+'/'+key,'size_bytes':len(data),'sha256':digest,'readback_sha256':rb,'backup_launch':launch},indent=2,sort_keys=True,default=str))

STATUS_READY
{
  "active_checkpoints": [],
  "active_run": null,
  "backup_launch": {
    "backup_dir": "/home/sagemaker-user/brain-mri-segmentation/evidence/study_v2/backups/20261108T134742680014Z",
    "pid": 926490,
    "runner_sha256": "8eacab0b3a8d6ef5dc413397425b45b916728d82496aef49b29924186ae02456",
    "stamp": "20261108T134742680014Z"
  },
  "captured_utc": "2026-11-08T13:47:42.618557+00:00",
  "checkpoint_check": {
    "path": "runs/study_v2/bce_dice_w2_s0/best.pt",
    "previous_manifest_match": null,
    "sha256": "f795b61915461ecf45602156dbef0e637e02f6290254da7a9e65e7535b0dddb9",
    "stable_hash": true
  },
  "completed_run_count": 14,
  "completed_run_names": [
    "bce_baseline_s0",
    "bce_baseline_s1",
    "bce_baseline_s2",
    "bce_dice_s0",
    "bce_dice_s1",
    "bce_dice_s2",
    "weighted_bce_s0",
    "weighted_bce_s1",
    "weighted_bce_s2",
    "bce_dice_posonly_s0",
    "bce_dice_posonly_s1",
    "bce_dice_posonly_s2",
    "bce_dice_w0.5_s0",
    "bce_dice_w

In [23]:
import json,subprocess,datetime
from pathlib import Path
B=Path('/home/sagemaker-user/brain-mri-segmentation/evidence/study_v2/backups/20261108T134742680014Z'); q=subprocess.run(['ps','-p','926490','-o','pid=,etime=,stat=,args='],capture_output=True,text=True); o={'backup_dir':str(B),'runner_process':q.stdout.strip() or None,'files':[{'name':p.name,'size_bytes':p.stat().st_size,'mtime_utc':datetime.datetime.fromtimestamp(p.stat().st_mtime,datetime.timezone.utc).isoformat()} for p in sorted(B.iterdir()) if p.is_file()]}; p=B/'snapshot_upload.log'; o['log_tail']=p.read_text(errors='replace').splitlines()[-20:] if p.exists() else []; p=B/'backup_receipt.json'; o['receipt']=json.loads(p.read_text()) if p.exists() else None; print('BACKUP_RUNNER_STATUS'); print(json.dumps(o,indent=2,sort_keys=True,default=str))

BACKUP_RUNNER_STATUS
{
  "backup_dir": "/home/sagemaker-user/brain-mri-segmentation/evidence/study_v2/backups/20261108T134742680014Z",
  "files": [
    {
      "mtime_utc": "2026-11-08T13:48:04.070917+00:00",
      "name": "backup_receipt.json",
      "size_bytes": 2946
    },
    {
      "mtime_utc": "2026-11-08T13:47:59.260704+00:00",
      "name": "brain-mri-segmentation-work.tar.gz",
      "size_bytes": 463862327
    },
    {
      "mtime_utc": "2026-11-08T13:47:43.950045+00:00",
      "name": "snapshot_manifest.json",
      "size_bytes": 199946
    },
    {
      "mtime_utc": "2026-11-08T13:47:42.669989+00:00",
      "name": "snapshot_started.json",
      "size_bytes": 306
    },
    {
      "mtime_utc": "2026-11-08T13:48:04.070917+00:00",
      "name": "snapshot_upload.log",
      "size_bytes": 2763
    }
  ],
  "log_tail": [
    "BACKUP_DONE {\"active_runs_at_start\": [], \"archive_local_path\": \"/home/sagemaker-user/brain-mri-segmentation/evidence/study_v2/backups/20261108T134

In [24]:
import json,hashlib,base64,tarfile,boto3
from pathlib import Path
B=Path('/home/sagemaker-user/brain-mri-segmentation/evidence/study_v2/backups/20261108T134742680014Z'); A=B/'brain-mri-segmentation-work.tar.gz'; r=json.loads((B/'backup_receipt.json').read_text()); m=json.loads((B/'snapshot_manifest.json').read_text())
h=hashlib.sha256()
with open(A,'rb') as f:
 for b in iter(lambda:f.read(1048576),b''): h.update(b)
sha=h.hexdigest(); tf=tarfile.open(A,'r:gz'); members=[x.name for x in tf.getmembers() if x.isfile()]; tf.close(); pref='brain-mri-segmentation/'; rel=[x[len(pref):] if x.startswith(pref) else x for x in members]; expected={x['path'] for x in m.get('included_files',[])}; mf=[x for x in rel if x.endswith('snapshot_manifest.json')]; datafiles=set(rel)-set(mf); missing=sorted(expected-datafiles); extra=sorted(datafiles-expected); s3=boto3.client('s3',region_name='us-east-1'); head=s3.head_object(Bucket=r['s3_bucket'],Key=r['s3_key'],ChecksumMode='ENABLED'); out={'status':r['status'],'archive_size':A.stat().st_size,'receipt_size':r['archive_size_bytes'],'size_match':A.stat().st_size==r['archive_size_bytes']==head['ContentLength'],'local_sha256':sha,'receipt_sha256':r['archive_sha256'],'head_metadata_sha256':head.get('Metadata',{}).get('sha256'),'sha_match':sha==r['archive_sha256']==head.get('Metadata',{}).get('sha256'),'head_checksum_base64':head.get('ChecksumSHA256'),'receipt_checksum_base64':r.get('s3_head_checksum_sha256_base64'),'checksum_match':head.get('ChecksumSHA256')==r.get('s3_head_checksum_sha256_base64') if head.get('ChecksumSHA256') else r.get('checksum_matches'),'tar_member_count':len(members),'manifest_records':len(expected),'manifest_member':mf,'missing_manifest_members':missing,'unlisted_archive_members':extra,'completed_run_count':m.get('completed_runs_at_start'),'run_names':[x.get('name') for x in m.get('run_summaries',[])],'active_runs':m.get('active_runs_at_start'),'stable_completed_checkpoints':r.get('completed_checkpoint_count'),'included_files':r.get('included_file_count'),'excluded_directories':m.get('excluded_directories'),'skipped_files':m.get('skipped_files'),'errors_found':m.get('errors_found'),'manifest_sha256':hashlib.sha256((B/'snapshot_manifest.json').read_bytes()).hexdigest(),'manifest_sha_receipt':r.get('snapshot_manifest_sha256')}; print('ARCHIVE_VERIFICATION'); print(json.dumps(out,indent=2,sort_keys=True,default=str))

ARCHIVE_VERIFICATION
{
  "active_runs": [],
  "archive_size": 463862327,
  "checksum_match": true,
  "completed_run_count": 14,
  "errors_found": [],
  "excluded_directories": [
    {
      "path": "data",
      "reason": "dataset_directory_excluded"
    },
    {
      "path": ".pytest_cache",
      "reason": "cache_or_vcs_excluded"
    },
    {
      "path": "evidence/study_v2/backups",
      "reason": "existing_or_current_backup_excluded"
    },
    {
      "path": "notebooks/.ipynb_checkpoints",
      "reason": "cache_or_vcs_excluded"
    },
    {
      "path": "scripts/__pycache__",
      "reason": "cache_or_vcs_excluded"
    },
    {
      "path": "src/lggseg/__pycache__",
      "reason": "cache_or_vcs_excluded"
    },
    {
      "path": "tests/.ipynb_checkpoints",
      "reason": "cache_or_vcs_excluded"
    },
    {
      "path": "tests/__pycache__",
      "reason": "cache_or_vcs_excluded"
    },
    {
      "path": "versions/study_v2/src/lggseg/__pycache__",
      "reason": "ca

In [25]:
import boto3,hashlib,tarfile,datetime,json,shutil
from pathlib import Path,PurePosixPath
R=Path('/home/sagemaker-user/brain-mri-segmentation'); BUCKET='ajay-mlp1-lgg-044450716277-use1'; KEY='studio/analysis_tools/20261108T1320Z/analysis_tools_20261108T1320Z.tar.gz'; expected='c93f5c9b4e8084b1e944dbe09669d3809acfec4eebc1d108ced74119680c121'; stamp=datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'); T=R/'evidence/study_v2/tools'/stamp; T.mkdir(parents=True,exist_ok=False)
s3=boto3.client('s3',region_name='us-east-1'); head=s3.head_object(Bucket=BUCKET,Key=KEY)
if head['ContentLength']!=6602: raise RuntimeError('unexpected tool archive size')
raw=s3.get_object(Bucket=BUCKET,Key=KEY)['Body'].read(); got=hashlib.sha256(raw).hexdigest()
if len(raw)!=6602 or got!=expected: raise RuntimeError('tool archive size/SHA-256 mismatch; extraction stopped')
arc=T/'analysis_tools_20261108T1320Z.tar.gz'; arc.write_bytes(raw); X=T/'extracted'; X.mkdir()
with tarfile.open(arc,'r:gz') as tf:
 members=tf.getmembers()
 for m in members:
  q=PurePosixPath(m.name)
  if q.is_absolute() or '..' in q.parts or not q.parts: raise RuntimeError('unsafe archive path: '+m.name)
  dest=X.joinpath(*q.parts)
  if m.isdir(): dest.mkdir(parents=True,exist_ok=True)
  elif m.isfile():
   dest.parent.mkdir(parents=True,exist_ok=True)
   with tf.extractfile(m) as src, open(dest,'xb') as dst: shutil.copyfileobj(src,dst)
  else: raise RuntimeError('non-regular archive member rejected: '+m.name)
files=sorted(p.relative_to(X).as_posix() for p in X.rglob('*') if p.is_file()); readmes=[p for p in X.rglob('README_PREP_v2.md')]; scripts=[p for p in X.rglob('val_analysis.py')]
if len(readmes)!=1 or len(scripts)!=1: raise RuntimeError('expected one README and one val_analysis.py')
readme=readmes[0].read_text(errors='replace'); receipt={'captured_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),'stamp':stamp,'s3_bucket':BUCKET,'s3_key':KEY,'archive_size_bytes':len(raw),'archive_sha256':got,'tool_dir':str(T),'extract_dir':str(X),'files':files,'readme_relative_path':readmes[0].relative_to(X).as_posix(),'script_relative_path':scripts[0].relative_to(X).as_posix()}; (T/'tool_receipt.json').write_text(json.dumps(receipt,indent=2,sort_keys=True)); print('TOOL_FETCH_VERIFIED'); print(json.dumps(receipt,indent=2,sort_keys=True)); print('README_PREP_v2_BEGIN'); print(readme); print('README_PREP_v2_END')

RuntimeError: tool archive size/SHA-256 mismatch; extraction stopped

In [26]:
import boto3,hashlib,tarfile,datetime,json,shutil
from pathlib import Path,PurePosixPath
R=Path('/home/sagemaker-user/brain-mri-segmentation'); BUCKET='ajay-mlp1-lgg-044450716277-use1'; KEY='studio/analysis_tools/20261108T1320Z/analysis_tools_20261108T1320Z.tar.gz'; expected='1c93f5c9b4e8084b1e944dbe09669d3809acfec4eebc1d108ced74119680c121'
if len(expected)!=64: raise RuntimeError('expected SHA-256 length is not 64: '+str(len(expected)))
s3=boto3.client('s3',region_name='us-east-1'); head=s3.head_object(Bucket=BUCKET,Key=KEY); raw=s3.get_object(Bucket=BUCKET,Key=KEY)['Body'].read(); got=hashlib.sha256(raw).hexdigest()
if len(raw)!=6602 or got!=expected:
 print(json.dumps({'head_size_bytes':head['ContentLength'],'actual_size_bytes':len(raw),'expected_size_bytes':6602,'actual_sha256':got,'expected_sha256':expected},indent=2)); raise RuntimeError('tool archive size/SHA-256 mismatch; extraction stopped')
stamp=datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'); T=R/'evidence/study_v2/tools'/stamp; T.mkdir(parents=True,exist_ok=False); arc=T/'analysis_tools_20261108T1320Z.tar.gz'; arc.write_bytes(raw); X=T/'extracted'; X.mkdir()
with tarfile.open(arc,'r:gz') as tf:
 for m in tf.getmembers():
  q=PurePosixPath(m.name)
  if q.is_absolute() or '..' in q.parts or not q.parts: raise RuntimeError('unsafe archive path: '+m.name)
  dest=X.joinpath(*q.parts)
  if m.isdir(): dest.mkdir(parents=True,exist_ok=True)
  elif m.isfile():
   dest.parent.mkdir(parents=True,exist_ok=True)
   with tf.extractfile(m) as src, open(dest,'xb') as dst: shutil.copyfileobj(src,dst)
  else: raise RuntimeError('non-regular archive member rejected: '+m.name)
files=sorted(p.relative_to(X).as_posix() for p in X.rglob('*') if p.is_file()); rd=list(X.rglob('README_PREP_v2.md')); sc=list(X.rglob('val_analysis.py'))
if len(rd)!=1 or len(sc)!=1: raise RuntimeError('expected one README and one val_analysis.py')
readme=rd[0].read_text(errors='replace'); rec={'stamp':stamp,'s3_key':KEY,'archive_size_bytes':len(raw),'archive_sha256':got,'tool_dir':str(T),'extract_dir':str(X),'files':files,'readme_relative_path':rd[0].relative_to(X).as_posix(),'script_relative_path':sc[0].relative_to(X).as_posix()}; (T/'tool_receipt.json').write_text(json.dumps(rec,indent=2,sort_keys=True)); print('TOOL_FETCH_VERIFIED'); print(json.dumps(rec,indent=2,sort_keys=True)); print('README_PREP_v2_BEGIN'); print(readme); print('README_PREP_v2_END')

TOOL_FETCH_VERIFIED
{
  "archive_sha256": "1c93f5c9b4e8084b1e944dbe09669d3809acfec4eebc1d108ced74119680c121",
  "archive_size_bytes": 6602,
  "extract_dir": "/home/sagemaker-user/brain-mri-segmentation/evidence/study_v2/tools/20261108T135252289825Z/extracted",
  "files": [
    "analysis_tools_20261108T1320Z/docs/README_PREP_v2.md",
    "analysis_tools_20261108T1320Z/docs/future_cells_DO_NOT_RUN.v2.py",
    "analysis_tools_20261108T1320Z/val_tools/tests/test_val_analysis.py",
    "analysis_tools_20261108T1320Z/val_tools/val_analysis.py"
  ],
  "readme_relative_path": "analysis_tools_20261108T1320Z/docs/README_PREP_v2.md",
  "s3_key": "studio/analysis_tools/20261108T1320Z/analysis_tools_20261108T1320Z.tar.gz",
  "script_relative_path": "analysis_tools_20261108T1320Z/val_tools/val_analysis.py",
  "stamp": "20261108T135252289825Z",
  "tool_dir": "/home/sagemaker-user/brain-mri-segmentation/evidence/study_v2/tools/20261108T135252289825Z"
}
README_PREP_v2_BEGIN
# Prep materials v2 — 2026-11-

In [27]:
import json,hashlib
from pathlib import Path
R=Path('/home/sagemaker-user/brain-mri-segmentation'); T=R/'evidence/study_v2/tools/20261108T135252289825Z'; rec=json.loads((T/'tool_receipt.json').read_text()); p=T/'extracted'/rec['script_relative_path']; text=p.read_text(); print('SCRIPT_AUDIT',json.dumps({'path':str(p),'sha256':hashlib.sha256(text.encode()).hexdigest(),'lines':len(text.splitlines()),'bytes':len(text.encode())})); print(text)

SCRIPT_AUDIT {"path": "/home/sagemaker-user/brain-mri-segmentation/evidence/study_v2/tools/20261108T135252289825Z/extracted/analysis_tools_20261108T1320Z/val_tools/val_analysis.py", "sha256": "a019fd54a29cd2b440fc319b9d3aafffe22bb09937ab3fed54862a23552e6ae0", "lines": 106, "bytes": 6758}
"""Validation-only post-hoc analysis for study_v2 runs (NEW separate utility; does not modify trained source).
Uses ONLY train/val manifests, each run's best.pt, channel_stats.json and run_meta.json (frozen per-run threshold).
Never reads test files. Outputs: per-run slice CSV, recompute check vs run_meta, lesion-size groups (train-derived
tertile cutoffs on positive-slice foreground_pixels), per-patient pooled Dice, paired per-patient differences
(seed-matched and seed-averaged), optional overlays.
Usage (Studio, after all training finishes):
  PYTHONPATH=~/brain-mri-segmentation/versions/study_v2/src python val_analysis.py \
    --data-root ~/brain-mri-segmentation/data --runs-root ~/brain-mri-segmen

In [28]:
import json,hashlib,datetime,subprocess,sys,os
from pathlib import Path
R=Path('/home/sagemaker-user/brain-mri-segmentation'); T=R/'evidence/study_v2/tools/20261108T135252289825Z'; rec=json.loads((T/'tool_receipt.json').read_text()); script=T/'extracted'/rec['script_relative_path']; digest=hashlib.sha256(script.read_bytes()).hexdigest(); expected='a019fd54a29cd2b440fc319b9d3aafffe22bb09937ab3fed54862a23552e6ae0'
if digest!=expected: raise RuntimeError('prepared analysis script hash changed; stopped')
stamp=datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'); OUT=R/'evidence/study_v2'/('val_analysis_bce_vs_bcedice_'+stamp)
if OUT.exists(): raise RuntimeError('analysis output path already exists; stopped')
OUT.mkdir(parents=True,exist_ok=False); log=R/'evidence/study_v2'/(OUT.name+'.log'); receipt=OUT/'launch_receipt.json'; env=os.environ.copy(); env['PYTHONPATH']=str(R/'versions/study_v2/src')
cmd=[sys.executable,str(script),'--data-root',str(R/'data'),'--runs-root',str(R/'runs/study_v2'),'--out',str(OUT),'--a','bce_baseline','--b','bce_dice','--seeds','0','1','2','--overlays']
launch={'started_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),'pid':None,'script':str(script),'script_sha256':digest,'pythonpath':env['PYTHONPATH'],'cwd':str(R),'command':cmd,'output_dir':str(OUT),'log':str(log),'tool_archive_sha256':rec['archive_sha256']}
f=open(log,'ab',buffering=0); p=subprocess.Popen(cmd,cwd=str(R),env=env,stdin=subprocess.DEVNULL,stdout=f,stderr=subprocess.STDOUT,start_new_session=True,close_fds=True); launch['pid']=p.pid; receipt.write_text(json.dumps(launch,indent=2,sort_keys=True)); print('VAL_ANALYSIS_LAUNCHED'); print(json.dumps(launch,indent=2,sort_keys=True))

VAL_ANALYSIS_LAUNCHED
{
  "command": [
    "/opt/conda/bin/python",
    "/home/sagemaker-user/brain-mri-segmentation/evidence/study_v2/tools/20261108T135252289825Z/extracted/analysis_tools_20261108T1320Z/val_tools/val_analysis.py",
    "--data-root",
    "/home/sagemaker-user/brain-mri-segmentation/data",
    "--runs-root",
    "/home/sagemaker-user/brain-mri-segmentation/runs/study_v2",
    "--out",
    "/home/sagemaker-user/brain-mri-segmentation/evidence/study_v2/val_analysis_bce_vs_bcedice_20261108T135340746642Z",
    "--a",
    "bce_baseline",
    "--b",
    "bce_dice",
    "--seeds",
    "0",
    "1",
    "2",
    "--overlays"
  ],
  "cwd": "/home/sagemaker-user/brain-mri-segmentation",
  "log": "/home/sagemaker-user/brain-mri-segmentation/evidence/study_v2/val_analysis_bce_vs_bcedice_20261108T135340746642Z.log",
  "output_dir": "/home/sagemaker-user/brain-mri-segmentation/evidence/study_v2/val_analysis_bce_vs_bcedice_20261108T135340746642Z",
  "pid": 926703,
  "pythonpath": "/ho

In [29]:
import json,subprocess,datetime
from pathlib import Path
R=Path('/home/sagemaker-user/brain-mri-segmentation'); O=R/'evidence/study_v2/val_analysis_bce_vs_bcedice_20261108T135340746642Z'; L=R/'evidence/study_v2/val_analysis_bce_vs_bcedice_20261108T135340746642Z.log'; q=subprocess.run(['ps','-p','926703','-o','pid=,etime=,stat=,args='],capture_output=True,text=True); files=[{'name':p.name,'size_bytes':p.stat().st_size} for p in sorted(O.iterdir())] if O.exists() else []; summary=json.loads((O/'summary.json').read_text()) if (O/'summary.json').exists() else None; log=L.read_text(errors='replace').splitlines()[-80:] if L.exists() else []; print('VAL_ANALYSIS_STATUS'); print(json.dumps({'captured_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),'pid':926703,'process':q.stdout.strip() or None,'output_dir':str(O),'files':files,'summary':summary,'log_tail':log},indent=2,sort_keys=True,default=str))

VAL_ANALYSIS_STATUS
{
  "captured_utc": "2026-11-08T13:54:17.769965+00:00",
  "files": [
    {
      "name": "bce_baseline_s0_overlay.png",
      "size_bytes": 247834
    },
    {
      "name": "bce_baseline_s0_val_slices.csv",
      "size_bytes": 46975
    },
    {
      "name": "bce_baseline_s1_overlay.png",
      "size_bytes": 258452
    },
    {
      "name": "bce_baseline_s1_val_slices.csv",
      "size_bytes": 46905
    },
    {
      "name": "bce_baseline_s2_overlay.png",
      "size_bytes": 273884
    },
    {
      "name": "bce_baseline_s2_val_slices.csv",
      "size_bytes": 46951
    },
    {
      "name": "bce_dice_s0_overlay.png",
      "size_bytes": 251179
    },
    {
      "name": "bce_dice_s0_val_slices.csv",
      "size_bytes": 44467
    },
    {
      "name": "bce_dice_s1_overlay.png",
      "size_bytes": 262326
    },
    {
      "name": "bce_dice_s1_val_slices.csv",
      "size_bytes": 44289
    },
    {
      "name": "bce_dice_s2_overlay.png",
      "size_bytes": 

In [30]:
import json,hashlib,tarfile,datetime,boto3
from pathlib import Path
R=Path('/home/sagemaker-user/brain-mri-segmentation'); T=R/'evidence/study_v2/tools/20261108T135252289825Z'; O=R/'evidence/study_v2/val_analysis_bce_vs_bcedice_20261108T135340746642Z'; NB=R/'notebooks/study_v2_preflight.ipynb'; LOG=R/'evidence/study_v2/val_analysis_bce_vs_bcedice_20261108T135340746642Z.log'; summary=json.loads((O/'summary.json').read_text()); stamp=datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'); D=R/'evidence/study_v2/analysis_packages'/stamp; D.mkdir(parents=True,exist_ok=False); A=D/'val_analysis_bce_vs_bcedice.tar.gz'
if not NB.exists() or not (O/'summary.json').exists(): raise RuntimeError('required notebook or results missing')
with tarfile.open(A,'w:gz',compresslevel=1) as tf:
 tf.add(O,arcname='analysis_outputs/'+O.name,recursive=True); tf.add(T,arcname='tool/'+T.name,recursive=True); tf.add(NB,arcname='notebooks/study_v2_preflight.ipynb'); tf.add(LOG,arcname='logs/'+LOG.name)
h=hashlib.sha256()
with open(A,'rb') as f:
 for b in iter(lambda:f.read(1048576),b''): h.update(b)
digest=h.hexdigest(); bucket='ajay-mlp1-lgg-044450716277-use1'; key='studio/analysis_results/'+stamp+'/val_analysis_bce_vs_bcedice.tar.gz'; s3=boto3.client('s3',region_name='us-east-1'); s3.put_object(Bucket=bucket,Key=key,Body=A.read_bytes(),ContentType='application/gzip',Metadata={'sha256':digest}); head=s3.head_object(Bucket=bucket,Key=key); body=s3.get_object(Bucket=bucket,Key=key)['Body']; rb=hashlib.sha256(body.read()).hexdigest(); body.close()
if head['ContentLength']!=A.stat().st_size or head.get('Metadata',{}).get('sha256')!=digest or rb!=digest: raise RuntimeError('analysis package S3 verification failed')
receipt={'created_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),'local_archive':str(A),'s3_uri':'s3://'+bucket+'/'+key,'size_bytes':A.stat().st_size,'sha256':digest,'readback_sha256':rb,'included':['analysis outputs','verified tool archive and extracted tool files','saved preflight notebook','analysis log'],'excluded':['trained checkpoints','dataset files','test/holdout artifacts']}; (D/'package_receipt.json').write_text(json.dumps(receipt,indent=2,sort_keys=True))
compact={n:{'threshold':v.get('threshold'),'pooled_foreground_dice':v['summary'].get('pooled_foreground_dice'),'positive_slice_dice':v['summary'].get('positive_slice_dice'),'patient_dice_mean':v['summary'].get('patient_dice_mean'),'abs_diff_vs_run_meta':v.get('abs_diff_vs_run_meta')} for n,v in summary['runs'].items()}; paired={k:{z:v.get(z) for z in ('n_patients','mean_diff_b_minus_a','median_diff','n_b_better','n_a_better','n_tie')} for k,v in summary['paired'].items()}
print('ANALYSIS_PACKAGE_VERIFIED'); print(json.dumps({'receipt':receipt,'run_metrics':compact,'paired_patient_differences':paired,'cutoffs_train':summary.get('cutoffs_train'),'data_scope':summary.get('data')},indent=2,sort_keys=True,default=str))

ANALYSIS_PACKAGE_VERIFIED
{
  "cutoffs_train": [
    1037.999999999998,
    2350.666666666665
  ],
  "data_scope": "train/val manifests only; no test access",
  "paired_patient_differences": {
    "seed0": {
      "mean_diff_b_minus_a": -0.07372884774301619,
      "median_diff": -0.02928238489261592,
      "n_a_better": 12,
      "n_b_better": 4,
      "n_patients": 17,
      "n_tie": 1
    },
    "seed1": {
      "mean_diff_b_minus_a": -0.02079875586782873,
      "median_diff": -0.014535898396682345,
      "n_a_better": 11,
      "n_b_better": 5,
      "n_patients": 17,
      "n_tie": 1
    },
    "seed2": {
      "mean_diff_b_minus_a": -0.0127723716853274,
      "median_diff": -0.014154401178595122,
      "n_a_better": 11,
      "n_b_better": 5,
      "n_patients": 17,
      "n_tie": 1
    },
    "seed_mean": {
      "mean_diff_b_minus_a": -0.035766658432057466,
      "median_diff": -0.021969576375554967,
      "n_a_better": 14,
      "n_b_better": 2,
      "n_patients": 17,
      "n

In [31]:
import json,datetime,subprocess
from pathlib import Path
R=Path('/home/sagemaker-user/brain-mri-segmentation'); V=R/'versions/study_v2'; D=R/'data'; FZ=R/'evidence/study_v2/freeze/FROZEN.json'; FD=FZ.parent; OUT=R/'evidence/study_v2/test_eval'; names=['bce_baseline_s0','bce_baseline_s1','bce_baseline_s2','bce_dice_s0','bce_dice_s1','bce_dice_s2','weighted_bce_s0','weighted_bce_s1','weighted_bce_s2','bce_dice_posonly_s0','bce_dice_posonly_s1','bce_dice_posonly_s2','bce_dice_w0.5_s0','bce_dice_w2_s0']; errs=[]
freeze_files=sorted(str(p.relative_to(FD)) for p in FD.rglob('*') if p.is_file()) if FD.exists() else []; eval_files=sorted(str(p.relative_to(OUT)) for p in OUT.rglob('*') if p.is_file()) if OUT.exists() else []
if FZ.exists() or Path(str(FZ)+'.sha256').exists() or Path(str(FZ)+'.sha256.json').exists() or freeze_files: errs.append({'reason':'existing freeze file(s)','files':freeze_files})
if eval_files: errs.append({'reason':'existing test_eval file(s)','files':eval_files})
if (D/'test').exists(): errs.append({'reason':'DATA/test already exists'})
TP=D/'test_manifest.csv'; expected='c6921b163eb39716b216365e8bb4c838b81e252349ce1f44726e412ddbd2e174'; existing_test=None
if TP.exists():
 import hashlib
 h=hashlib.sha256(TP.read_bytes()).hexdigest(); existing_test={'size_bytes':TP.stat().st_size,'sha256':h}
 if existing_test['size_bytes']!=163249 or h!=expected: errs.append({'reason':'existing test_manifest raw hash mismatch','size_bytes':existing_test['size_bytes'],'sha256':h})
E=R/'evidence/study_v2/async_study_v2_20261108T105406343233Z'; ds=json.loads((E/'driver_status.json').read_text()); rows=ds.get('runs',[]); done=[x for x in rows if str(x.get('status','')).lower()=='completed' and x.get('returncode',x.get('exit_code'))==0]; got={str(x.get('name',x.get('run_name',''))) for x in done}
if ds.get('driver_status')!='completed_all' or len(got)!=14 or got!=set(names): errs.append({'reason':'driver not completed_all with exact 14 completed runs','driver_status':ds.get('driver_status'),'completed_count':len(got),'missing':sorted(set(names)-got),'unexpected':sorted(got-set(names))})
script=V/'scripts/freeze.py'; inv=[p.relative_to(V).as_posix() for p in V.rglob('*') if p.is_file() and any(k in p.name.lower() for k in ('inventory','hash','manifest'))]; checkpoint_stats=[]
for n in names:
 rd=R/'runs/study_v2'/n; meta=rd/'run_meta.json'; ck=rd/'best.pt'; checkpoint_stats.append({'name':n,'run_meta_exists':meta.exists(),'checkpoint_exists':ck.exists(),'checkpoint_size_bytes':ck.stat().st_size if ck.exists() else None,'checkpoint_mtime_ns':ck.stat().st_mtime_ns if ck.exists() else None})
if not script.is_file(): errs.append({'reason':'corrected freeze.py missing','path':str(script)})
if any(not x['run_meta_exists'] or not x['checkpoint_exists'] for x in checkpoint_stats): errs.append({'reason':'missing run metadata/checkpoint','runs':[x['name'] for x in checkpoint_stats if not x['run_meta_exists'] or not x['checkpoint_exists']]})
result={'checked_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),'root':str(R),'freeze_path':str(FZ),'freeze_exists':FZ.exists(),'freeze_sidecar_exists':Path(str(FZ)+'.sha256').exists(),'existing_freeze_files':freeze_files,'test_eval_path':str(OUT),'existing_test_eval_files':eval_files,'data_test_dir_exists':(D/'test').exists(),'test_manifest_path':str(TP),'test_manifest_preexisting_raw_metadata':existing_test,'train_val_manifests':[{'path':str(D/f'{x}_manifest.csv'),'exists':(D/f'{x}_manifest.csv').is_file(),'size_bytes':(D/f'{x}_manifest.csv').stat().st_size if (D/f'{x}_manifest.csv').is_file() else None} for x in ('train','val')],'driver_status':ds.get('driver_status'),'completed_run_count':len(got),'driver_pid':ds.get('driver_pid'),'versioned_freeze_script':str(script),'freeze_script_sha256':__import__('hashlib').sha256(script.read_bytes()).hexdigest() if script.exists() else None,'source_inventory_candidates':inv,'checkpoint_stats':checkpoint_stats,'halt_reasons':errs}; print('FREEZE_PREFLIGHT'); print(json.dumps(result,indent=2,sort_keys=True,default=str))
if errs: raise RuntimeError('freeze preflight failed; no manifest download or freeze attempted')

FREEZE_PREFLIGHT
{
  "checked_utc": "2026-11-08T13:59:11.014044+00:00",
  "checkpoint_stats": [
    {
      "checkpoint_exists": true,
      "checkpoint_mtime_ns": 1794136009265338028,
      "checkpoint_size_bytes": 31107871,
      "name": "bce_baseline_s0",
      "run_meta_exists": true
    },
    {
      "checkpoint_exists": true,
      "checkpoint_mtime_ns": 1794136711226920722,
      "checkpoint_size_bytes": 31107871,
      "name": "bce_baseline_s1",
      "run_meta_exists": true
    },
    {
      "checkpoint_exists": true,
      "checkpoint_mtime_ns": 1794137472311258250,
      "checkpoint_size_bytes": 31107871,
      "name": "bce_baseline_s2",
      "run_meta_exists": true
    },
    {
      "checkpoint_exists": true,
      "checkpoint_mtime_ns": 1794137915361199045,
      "checkpoint_size_bytes": 31107871,
      "name": "bce_dice_s0",
      "run_meta_exists": true
    },
    {
      "checkpoint_exists": true,
      "checkpoint_mtime_ns": 1794138952637542856,
      "checkpoint_s

In [32]:
from pathlib import Path
import hashlib
p=Path('/home/sagemaker-user/brain-mri-segmentation/versions/study_v2/scripts/freeze.py'); s=p.read_text(); print('FREEZE_SCRIPT',hashlib.sha256(p.read_bytes()).hexdigest(),len(s.splitlines()),'lines'); print(s)

FREEZE_SCRIPT 0fae33d53711a49722cb3df99c38c24b72ddc6dfd52c28e93ae73b8265890f72 6 lines
"""Freeze chosen runs BEFORE test evaluation: hashes checkpoints, normalization stats, run_meta, eval code."""
import argparse, os, sys
sys.path.insert(0, os.path.join(os.path.dirname(__file__), "..", "src"))
from lggseg.protocol import freeze
ap = argparse.ArgumentParser(); ap.add_argument("--runs", nargs="+", required=True); ap.add_argument("--out", default="FROZEN.json"); ap.add_argument("--manifests", nargs="*", default=[], help="train/val/test manifest CSVs to hash (test manifest: path/ID file only, hash only)")
a = ap.parse_args(); fr = freeze(a.runs, a.out, manifests=a.manifests); print("frozen", list(fr["runs"]), "->", a.out, "+ .sha256")



In [33]:
from pathlib import Path
R=Path('/home/sagemaker-user/brain-mri-segmentation'); V=R/'versions/study_v2'; lines=(V/'src/lggseg/protocol.py').read_text().splitlines(); i=next(i for i,x in enumerate(lines) if x.startswith('def freeze(')); j=next((j for j in range(i+1,len(lines)) if lines[j].startswith(('def ','class '))),len(lines)); print('FREEZE_FUNCTION_SOURCE'); print('
'.join(lines[max(0,i-8):j])); print('SOURCE_INVENTORY'); print((V/'INVENTORY.sha256').read_text()); print('MANIFEST_PATH',V/'docs/study_v2_execution_manifest.json')

SyntaxError: unterminated string literal (detected at line 2) (783465042.py, line 2)

In [34]:
from pathlib import Path
R=Path('/home/sagemaker-user/brain-mri-segmentation'); V=R/'versions/study_v2'; lines=(V/'src/lggseg/protocol.py').read_text().splitlines(); i=next(i for i,x in enumerate(lines) if x.startswith('def freeze(')); j=next((j for j in range(i+1,len(lines)) if lines[j].startswith(('def ','class '))),len(lines)); print('FREEZE_FUNCTION_SOURCE'); print(chr(10).join(lines[max(0,i-8):j])); print('SOURCE_INVENTORY'); print((V/'INVENTORY.sha256').read_text()); print('MANIFEST_PATH',V/'docs/study_v2_execution_manifest.json')

FREEZE_FUNCTION_SOURCE
def now():
    return time.strftime("%Y-%m-%dT%H:%M:%S%z")


class ProtocolError(SystemExit):
    pass


def freeze(runs, out, pkg_root=PKG, manifests=None):
    if os.path.exists(out):
        raise ProtocolError(f"{out} exists; freezing is one-shot.")
    fr = {"frozen_at": now(), "runs": {}, "code_sha256": {}, "manifest_sha256": {}}
    for m in (manifests or []):  # FIX: manifests (incl. test paths/IDs) are now hashed and re-verified
        fr["manifest_sha256"][os.path.abspath(m)] = sha256_file(m)
    for c in CODE_FILES:
        p = os.path.join(pkg_root, c)
        if os.path.exists(p): fr["code_sha256"][c] = sha256_file(p)
    for r in runs:
        meta = json.load(open(os.path.join(r, "run_meta.json")))
        if meta.get("smoke_test"): raise ProtocolError(f"{r} is a smoke test; cannot freeze")
        art = {a: sha256_file(os.path.join(r, a)) for a in RUN_ARTIFACTS}
        if meta.get("checkpoint_sha256") and meta["checkpoint_sha256"] != art["best.p

In [35]:
from pathlib import Path
p=Path('/home/sagemaker-user/brain-mri-segmentation/versions/study_v2/src/lggseg/protocol.py'); lines=p.read_text().splitlines(); print(chr(10).join(f'{i+1}: {x}' for i,x in enumerate(lines[:150])))

1: """Freeze + one-time test-evaluation protocol.
2: 
3: What this DOES guarantee: any change to frozen checkpoints, normalization statistics, run metadata,
4: configs, the freeze record, or the hashed source files is detected and evaluation refuses; a completed
5: or started test evaluation cannot be started again through this code path.
6: What it does NOT guarantee: it cannot stop someone editing this code, deleting the ledger, or computing
7: test metrics with another script. Integrity ultimately rests on the honesty of the experimenter and on
8: the append-only ledger + Google Doc log being disclosed in the report.
9: """
10: from __future__ import annotations
11: import glob, hashlib, json, os, socket, time, traceback
12: 
13: PKG = os.path.abspath(os.path.join(os.path.dirname(__file__), "..", ".."))
14: CODE_FILES = ["src/lggseg/data.py", "src/lggseg/model.py", "src/lggseg/metrics.py", "src/lggseg/protocol.py",
15:               "scripts/evaluate_test.py"]
16: RUN_ARTIFACTS = ["

In [36]:
from pathlib import Path
p=Path('/home/sagemaker-user/brain-mri-segmentation/versions/study_v2/src/lggseg/protocol.py'); lines=p.read_text().splitlines(); print(chr(10).join(f'{i+1}: {x}' for i,x in enumerate(lines[:95])))

1: """Freeze + one-time test-evaluation protocol.
2: 
3: What this DOES guarantee: any change to frozen checkpoints, normalization statistics, run metadata,
4: configs, the freeze record, or the hashed source files is detected and evaluation refuses; a completed
5: or started test evaluation cannot be started again through this code path.
6: What it does NOT guarantee: it cannot stop someone editing this code, deleting the ledger, or computing
7: test metrics with another script. Integrity ultimately rests on the honesty of the experimenter and on
8: the append-only ledger + Google Doc log being disclosed in the report.
9: """
10: from __future__ import annotations
11: import glob, hashlib, json, os, socket, time, traceback
12: 
13: PKG = os.path.abspath(os.path.join(os.path.dirname(__file__), "..", ".."))
14: CODE_FILES = ["src/lggseg/data.py", "src/lggseg/model.py", "src/lggseg/metrics.py", "src/lggseg/protocol.py",
15:               "scripts/evaluate_test.py"]
16: RUN_ARTIFACTS = ["

In [37]:
from pathlib import Path
p=Path('/home/sagemaker-user/brain-mri-segmentation/versions/study_v2/src/lggseg/protocol.py'); lines=p.read_text().splitlines(); print(chr(10).join(f'{i+1}: {x}' for i,x in enumerate(lines) if any(x.startswith(k) for k in ('PKG','CODE_FILES','RUN_ARTIFACTS'))))

13: PKG = os.path.abspath(os.path.join(os.path.dirname(__file__), "..", ".."))
14: CODE_FILES = ["src/lggseg/data.py", "src/lggseg/model.py", "src/lggseg/metrics.py", "src/lggseg/protocol.py",
16: RUN_ARTIFACTS = ["best.pt", "channel_stats.json", "run_meta.json"]


In [38]:
from pathlib import Path
p=Path('/home/sagemaker-user/brain-mri-segmentation/versions/study_v2/src/lggseg/protocol.py'); lines=p.read_text().splitlines(); print(chr(10).join(f'{i+1}: {x}' for i,x in enumerate(lines[:35])))

1: """Freeze + one-time test-evaluation protocol.
2: 
3: What this DOES guarantee: any change to frozen checkpoints, normalization statistics, run metadata,
4: configs, the freeze record, or the hashed source files is detected and evaluation refuses; a completed
5: or started test evaluation cannot be started again through this code path.
6: What it does NOT guarantee: it cannot stop someone editing this code, deleting the ledger, or computing
7: test metrics with another script. Integrity ultimately rests on the honesty of the experimenter and on
8: the append-only ledger + Google Doc log being disclosed in the report.
9: """
10: from __future__ import annotations
11: import glob, hashlib, json, os, socket, time, traceback
12: 
13: PKG = os.path.abspath(os.path.join(os.path.dirname(__file__), "..", ".."))
14: CODE_FILES = ["src/lggseg/data.py", "src/lggseg/model.py", "src/lggseg/metrics.py", "src/lggseg/protocol.py",
15:               "scripts/evaluate_test.py"]
16: RUN_ARTIFACTS = ["